# GameTheory 3b — Chambres, murs et chemins : la géométrie ordinale des jeux 2×2

**Navigation** : [Index](README.md) | [jumeau Python](GameTheory-03b-Chambres-Murs-Chemins-Lean-Python.ipynb)

**Navigation** : [Index](README.md) | [jumeau Rust](GameTheory-03b-Chambres-Murs-Chemins-Lean-Rust.ipynb)

> **Deuxième volet de la chaîne 03** — précédent : [GameTheory-03-Topology2x2-Python](GameTheory-03-Topology2x2-Python.ipynb) (les 576 jeux stricts et la grammaire des swaps) ; suivants : [Dynamics-10-Joueur-LLM-Python](Dynamics/Dynamics-10-Joueur-LLM-Python.ipynb) (la table périodique appliquée à un LLM), [GameTheory-03d-Deux-Especes-de-Fleches-Python](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) (quand un swap préserve les équilibres) et [GameTheory-03e-Meta-Actions-Tarifees-Python](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb) (le capstone).

Ce carnet décrit **l'espace** dans lequel les transformations de GT-3 se déplacent, puis **sa métrique** : les strates de l'espace des paiements et leur incidence (§1-§4), puis la distance de swap entre deux jeux et les chemins qui la réalisent (§5-§10). Les deux moitiés se lisent dans cet ordre — une distance se mesure en murs traversés, et les murs sont ce que la première moitié construit.

**La thèse, en une phrase** : un jeu à égalités n'est pas un jeu « plus flou » ou « plus général » qu'un jeu strict — c'est un objet **de dimension inférieure**, un mur où des chambres se touchent ; et la distance de swap entre deux jeux stricts est la somme des longueurs de Coxeter de leurs deux côtés, mesurée par le nombre de murs traversés.

La lecture géométrique vient de la formalisation de **Bruns & Kimmich** (rapportée, cf. §Sources) :

- les jeux **stricts** — aucune égalité de paiements — sont les **chambres** : les composantes ouvertes de l'espace des paiements ;
- les jeux **à égalités** sont les **murs** : des strates de **codimension ≥ 1** sur lesquelles plusieurs chambres se raccordent ;
- une transformation élémentaire n'est pas un saut abstrait d'un jeu à un autre : elle **traverse une face**.

```
chambre  →  mur  →  chambre voisine
```

Un « objet botanique » — un jeu à égalités depuis lequel plusieurs prolongements sont possibles — n'est donc **pas** une définition assouplie : c'est **habiter le bord**. La différence entre les deux notions est la différence entre *relâcher un axiome* et *se placer sur une strate de codimension 1*.

**Objectifs d'apprentissage** — à la fin de ce carnet, vous savez :

1. énumérer les strates de l'espace des paiements et les classer par codimension (75 ordres faibles par joueur, 5 625 jeux à rangs) ;
2. mesurer l'incidence chambre-mur et lire le graphe des chambres (connexité, diamètre, distribution des distances) ;
3. calculer une distance de swap par parcours en largeur, et énumérer les plus courts chemins entre deux jeux ;
4. vérifier le théorème de décomposition $d(G,H) = \ell(r_G,r_H) + \ell(c_G,c_H)$ par énumération exhaustive, et distinguer un chemin **valide** d'un chemin **minimal** — le premier se certifie, le second se mesure.

**Prérequis** : [GameTheory-03-Topology2x2-Python](GameTheory-03-Topology2x2-Python.ipynb) — les 576 jeux stricts et les six générateurs $R_{12}/R_{23}/R_{34}/C_{12}/C_{23}/C_{34}$. Python 3.10+, aucune dépendance externe.

**Durée estimée** : 1 h 30 de lecture active, plus les six exercices.

**Plan** : §1 les strates classées par codimension · §2 l'incidence chambre-mur et le graphe des chambres · §3 traverser un mur, puis relire la grammaire des swaps en longueurs de Coxeter · §4 *make tie* / *break tie* et trois archétypes · §5 l'espace des 576 jeux et ses générateurs · §6 la distance d'un jeu à l'autre — BFS depuis le Dilemme, tableau des distances aux archétypes, les trois cousins à distance 2 · §7 la distribution des distances — sphères emboîtées, symétrie parfaite, convolution des distributions d'inversions de $S_4$ · §8 le théorème de décomposition — la distance est la somme des longueurs de Coxeter des deux côtés, vérifié par énumération exhaustive sur les 331 776 paires · §9 les multiplicités des plus courts chemins — de 2 (commutation) à 236 544 (l'anti-Dilemme) · §10 générateur contre certificat — le miroir Rust du vérificateur Lean, valide ≠ minimal, puis le module `Swaps/Basic.lean` lui-même, cité aux lignes mesurées · **Exercices** (6) — énumérer les murs par codimension, exhiber un triplet chambre-mur-chambre, make tie / break tie vers les archétypes, puis distance par sphères, diamètre depuis une autre source et certificat d'un chemin fourni.

**Conventions** : un jeu 2×2 ordinal = deux tables de paiements (Ligne, Colonne), chacune un 4-uplet $(r_{11}, r_{12}, r_{21}, r_{22})$ de valeurs ; un jeu strict = chaque table est une permutation de $\{1,2,3,4\}$.

## 0. Configuration

Notebook **bibliothèque standard seule** (`std`, sans crate) — aucune dépendance. Comme GT-21, tout ce qui est affirmé ici est **calculé dans ce notebook** ; ce qui vient de la littérature sans dérivation locale est **rapporté comme dette** (§Sources).

In [2]:
{
// GameTheory 3b : chambres, murs et chemins -- bibliothèque standard seule

println!("GameTheory 3b : chambres, murs et chemins");
println!("Representation : jeu = (table_ligne, table_colonne), tables = ordres faibles canoniques");
}


GameTheory 3b : chambres, murs et chemins


Representation : jeu = (table_ligne, table_colonne), tables = ordres faibles canoniques


()

## 1. L'espace des paiements et ses strates

### 1.1 Ordres faibles : la clé canonique

L'espace des paiements d'un joueur est $\mathbb{R}^4$ (une coordonnée par case de sa table 2×2). Ce qui compte pour un jeu ordinal n'est pas la valeur numérique d'une coordonnée mais son **rang** relativement aux trois autres. Un point de cet espace code donc un **ordre faible** (*weak order*) sur les 4 cases : soit un ordre total (les 4 valeurs distinctes — une chambre), soit un ordre avec des ex æquo (un mur).

La représentation canonique : un 4-uplet $t$ où $t_i = 1 + (\text{nombre de valeurs distinctes strictement inférieures à } t_i)$. Ainsi $(2,2,4,5)$, $(1,1,3,4)$ et $(7,7,8,9)$ codent le même ordre faible — deux cases liées en bas, deux cases liées au-dessus — et partagent la clé canonique $(1,1,2,3)$.

Le discret $\{1,2,3,4\}$ de GT-3/GT-21 est le cas particulier sans ex æquo : les 24 permutations de $(1,2,3,4)$.

In [3]:
{
// === Section 1.1 : enumeration des ordres faibles sur 4 cases ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn nb_blocs(t: Tab) -> usize {
    let mut v = tab_vals(t);
    v.sort();
    v.dedup();
    v.len()
}

let mut ensemble: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize {
    for b in 1..=4usize {
        for c in 1..=4usize {
            for d in 1..=4usize {
                ensemble.insert(canonique((a, b, c, d)));
            }
        }
    }
}
let ordres_faibles: Vec<Tab> = ensemble.into_iter().collect();
let n_stricts = ordres_faibles.iter().filter(|t| nb_blocs(**t) == 4).count();
println!("Ordres faibles sur 4 cases : {}", ordres_faibles.len());
println!("  stricts (chambres potentielles) : {}", n_stricts);
println!("  avec ex aequo (murs potentiels)  : {}", ordres_faibles.len() - n_stricts);
}


Ordres faibles sur 4 cases : 75


  stricts (chambres potentielles) : 24


  avec ex aequo (murs potentiels)  : 51


()

**Lecture de la sortie committée** : **75 ordres faibles** — c'est le *nombre de Fubini* $a(4)$, le compte des partitions ordonnées d'un ensemble à 4 éléments. Ils se partagent en **24 ordres totaux** (les permutations — les chambres au sens strict, inchangées depuis GT-3) et **51 ordres avec ex æquo** (les murs, par joueur).

Ce simple compte situe le changement de regard : le discret de GT-3 ($24^2 = 576$ jeux stricts) vivait dans un espace où les égalités étaient **exclues par construction**. En admettant les ex æquo dans la représentation, l'espace passe de $576$ à $75^2 = 5625$ jeux à rangs — et ces nouveaux points ne sont pas des « jeux imprécis » : ce sont des **strates**. Reste à leur donner une dimension.

### 1.2 Codimension : compter des égalités *indépendantes*

La codimension d'une strate est le nombre d'**équations indépendantes** qui la définissent — la dimension perdue. Pour un ordre faible, ce n'est pas le nombre de *paires* égales mais le **complément du nombre de blocs** :

$$\text{codim}(t) = 4 - \#\text{blocs}(t).$$

Un bloc de taille 3 (triple égalité) porte **3 paires** égales mais seulement **2 équations indépendantes** ($x_a = x_b$ et $x_b = x_c$ suffisent, la troisième s'en déduit). Confondre paires et équations est l'erreur classique — le code suivant les compte séparément pour montrer qu'elles divergent.

In [4]:
{
// === Section 1.2 : codimension vraie vs nombre de paires egales ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);

// str(tuple) de Python pour les signatures de blocs : (4,) ; (3, 1) ; (1, 1, 1, 1)
fn py_tup(v: &[usize]) -> String {
    let parts: Vec<String> = v.iter().map(|x| x.to_string()).collect();
    if v.len() == 1 {
        format!("({},)", parts[0])
    } else {
        format!("({})", parts.join(", "))
    }
}

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn nb_blocs(t: Tab) -> usize {
    let mut v = tab_vals(t);
    v.sort();
    v.dedup();
    v.len()
}

// effectifs des blocs, tries decroissant (= tuple(sorted(Counter(t).values(), reverse=True)))
fn blocs(t: Tab) -> Vec<usize> {
    let mut sorted = tab_vals(t);
    sorted.sort();
    let mut counts: Vec<usize> = Vec::new();
    let mut i = 0;
    while i < sorted.len() {
        let mut j = i;
        while j < sorted.len() && sorted[j] == sorted[i] {
            j += 1;
        }
        counts.push(j - i);
        i = j;
    }
    counts.sort_unstable();
    counts.reverse();
    counts
}

fn nb_paires_egales(t: Tab) -> usize {
    blocs(t).iter().map(|&m| m * (m - 1) / 2).sum()
}

fn codim(t: Tab) -> usize {
    4 - nb_blocs(t)
}

let mut ensemble: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize {
    for b in 1..=4usize {
        for c in 1..=4usize {
            for d in 1..=4usize {
                ensemble.insert(canonique((a, b, c, d)));
            }
        }
    }
}
let ordres_faibles: Vec<Tab> = ensemble.into_iter().collect();

println!("Type de blocs           | codim | paires | exemples");
println!("{}", "-".repeat(64));
let mut par_type: BTreeMap<Vec<usize>, Vec<Tab>> = BTreeMap::new();
for &t in &ordres_faibles {
    par_type.entry(blocs(t)).or_default().push(t);
}
let mut sigs: Vec<&Vec<usize>> = par_type.keys().collect();
sigs.sort();
sigs.reverse();
for sig in sigs {
    let ex = &par_type[sig as &Vec<usize>];
    let paires_moy: usize = ex.iter().map(|&t| nb_paires_egales(t)).sum::<usize>() / ex.len();
    let mut exemples = py_tup(&[ex[0].0, ex[0].1, ex[0].2, ex[0].3]);
    if ex.len() > 1 {
        exemples.push_str(", ");
        exemples.push_str(&py_tup(&[ex[1].0, ex[1].1, ex[1].2, ex[1].3]));
    }
    let sig_str = py_tup(sig);
    println!(
        "blocs {:10} ({:2} ordres) |   {}   |   {}    | {}",
        sig_str,
        ex.len(),
        4 - sig.len(),
        paires_moy,
        exemples
    );
}
println!();
let mut dist_codim: BTreeMap<usize, usize> = BTreeMap::new();
for &t in &ordres_faibles {
    *dist_codim.entry(codim(t)).or_default() += 1;
}
let somme: usize = dist_codim.values().sum();
println!("Distribution par codimension (par joueur) : {:?}", dist_codim);
println!("Verification de la somme : 24 + 36 + 14 + 1 = {} = 75 ordres faibles", somme);
}


Type de blocs           | codim | paires | exemples


----------------------------------------------------------------


blocs (4,)       ( 1 ordres) |   3   |   6    | (1, 1, 1, 1)


blocs (3, 1)     ( 8 ordres) |   2   |   3    | (1, 1, 1, 2), (1, 1, 2, 1)


blocs (2, 2)     ( 6 ordres) |   2   |   2    | (1, 1, 2, 2), (1, 2, 1, 2)


blocs (2, 1, 1)  (36 ordres) |   1   |   1    | (1, 1, 2, 3), (1, 1, 3, 2)


blocs (1, 1, 1, 1) (24 ordres) |   0   |   0    | (1, 2, 3, 4), (1, 2, 4, 3)


Distribution par codimension (par joueur) : {0: 24, 1: 36, 2: 14, 3: 1}


Verification de la somme : 24 + 36 + 14 + 1 = 75 = 75 ordres faibles


()

**Lecture de la sortie committée** : la taxonomy par taille de blocs croise les deux mesures.

| blocs | codim | paires égales | compte |
|---|---|---|---|
| $(1,1,1,1)$ | 0 | 0 | 24 — **les chambres** |
| $(2,1,1)$ | 1 | 1 | 36 — **les murs simples** |
| $(2,2)$ | 2 | 2 | 6 |
| $(3,1)$ | **2** | **3** | 8 |
| $(4)$ | **3** | **6** | 1 |

Les deux lignes en **gras** portent la leçon : le triple ex æquo (les 8 cas $(3,1)$) **diverge** — 3 paires mais codimension 2 — et le quadruple ex æquo porte 6 paires pour codimension 3. La codimension est l'auche correcte : elle mesure la **dimension perdue**, pas le nombre de coïncidences.

L'image botanique se précise : les murs de codimension 1 sont des **cloisons** (36 par joueur), ceux de codimension 2 des **arêtes** où les cloisons se rencontrent (6+8 = 14), celui de codimension 3 un **sommet** (1). Un objet botanique n'est pas « un jeu imprécis » : c'est un point de l'espace des paiements situé **sur** cette architecture.

### 1.3 Les 5 625 jeux à rangs, par codimension

Un jeu complet = une paire d'ordres faibles (Ligne, Colonne). La codimension du jeu est la somme des deux : un mur de Ligne dans une chambre de Colonne est un mur de codim 1 ; deux murs simultanés donnent codim 2 ; etc.

In [5]:
{
// === Section 1.3 : distribution des jeux par codimension totale ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn nb_blocs(t: Tab) -> usize {
    let mut v = tab_vals(t);
    v.sort();
    v.dedup();
    v.len()
}

fn codim(t: Tab) -> usize {
    4 - nb_blocs(t)
}

let mut ensemble: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize {
    for b in 1..=4usize {
        for c in 1..=4usize {
            for d in 1..=4usize {
                ensemble.insert(canonique((a, b, c, d)));
            }
        }
    }
}
let ordres_faibles: Vec<Tab> = ensemble.into_iter().collect();

let mut dist_jeux: BTreeMap<usize, usize> = BTreeMap::new();
for &r in &ordres_faibles {
    for &c in &ordres_faibles {
        *dist_jeux.entry(codim(r) + codim(c)).or_default() += 1;
    }
}
println!("Jeux a rangs (paires d'ordres faibles) : {}", ordres_faibles.len() * ordres_faibles.len());
println!();
println!("codim |  jeux  | remarque");
println!("{}", "-".repeat(56));
for (&k, &n) in &dist_jeux {
    let remarque = match k {
        0 => "les 576 chambres strictes (univers GT-3 / GT-21)",
        1 => "murs simples : un seul ex aequo, d'un seul cote",
        2 => "murs doubles : deux ex aequo (eventuellement croises)",
        6 => "les deux tables entierement plates (jeu unique)",
        _ => "",
    };
    println!("  {}   | {:5}  | {}", k, n, remarque);
}
println!();
let total_jeux: usize = dist_jeux.values().sum();
println!("Total : {} = 75 x 75 attendu", total_jeux);
println!("Chambres strictes : {} -- coherentes avec GT-3 (576) et GT-21", dist_jeux[&0]);
}


Jeux a rangs (paires d'ordres faibles) : 5625


codim |  jeux  | remarque


--------------------------------------------------------


  0   |   576  | les 576 chambres strictes (univers GT-3 / GT-21)


  1   |  1728  | murs simples : un seul ex aequo, d'un seul cote


  2   |  1968  | murs doubles : deux ex aequo (eventuellement croises)


... (3 outputs left out)

  6   |     1  | les deux tables entierement plates (jeu unique)


Total : 5625 = 75 x 75 attendu


Chambres strictes : 576 -- coherentes avec GT-3 (576) et GT-21


()

**Lecture de la sortie committée** : l'univers des jeux à rangs compte **5 625** états. Les **576 chambres strictes** — tout l'univers de GT-3 et de GT-21 — n'en occupent que **10,2 %**. Les **1 728 jeux de codimension 1** sont les murs simples ; **1 968** de codim 2 ; le reste s'étage jusqu'au jeu totalement plat (codim 6), unique, où chaque joueur est indifférent entre les quatre cases.

Autrement dit : le discret dans lequel la série a travaillé jusqu'ici était le **squelette ouvert** d'un édifice dont les murs, arêtes et sommets — 89,8 % des états à rangs — n'avaient pas de nom. La suite leur donne un rôle structurel : ce sont eux qui **raccordent** les chambres.

## 2. L'incidence chambre-mur

### 2.1 Chaque mur simple sépare exactement deux chambres

Un mur de codimension 1 (un seul ex æquo, d'un seul côté) est une cloison : la briser dans un sens ou dans l'autre donne **exactement deux** chambres. Réciproquement, depuis une chambre, quels murs touche-t-on ? Pas tous : pour que deux cases puissent **se rencontrer**, il faut qu'aucune autre valeur ne soit strictement comprise entre les leurs — sinon le mouvement croiserait d'autres murs d'abord. Les facettes d'une chambre correspondent aux paires de **valeurs adjacentes** dans son ordre.

Le code suivant mesure l'incidence dans les deux directions.

In [6]:
{
// === Section 2.1 : incidence double-face chambre <-> mur ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn nb_blocs(t: Tab) -> usize {
    let mut v = tab_vals(t);
    v.sort();
    v.dedup();
    v.len()
}

fn codim(t: Tab) -> usize {
    4 - nb_blocs(t)
}

fn briser_le_tie(t: Tab) -> (Tab, Tab) {
    let v = tab_vals(t);
    let mut sorted = v.clone();
    sorted.sort();
    let mut v_tie = 0;
    let mut i = 0;
    while i < sorted.len() {
        let mut j = i;
        while j < sorted.len() && sorted[j] == sorted[i] {
            j += 1;
        }
        if j - i == 2 {
            v_tie = sorted[i];
        }
        i = j;
    }
    let ties: Vec<usize> = (0..4).filter(|&p| v[p] == v_tie).collect();
    let (i2, j2) = (ties[0], ties[1]);
    let mut below: Vec<usize> = (0..4).filter(|&p| v[p] < v_tie).collect();
    below.sort_by_key(|&p| v[p]);
    let mut above: Vec<usize> = (0..4).filter(|&p| v[p] > v_tie).collect();
    above.sort_by_key(|&p| v[p]);
    let rangs = |ordre: Vec<usize>| -> Tab {
        let mut rk = [0usize; 4];
        for (k, &p) in ordre.iter().enumerate() {
            rk[p] = k + 1;
        }
        (rk[0], rk[1], rk[2], rk[3])
    };
    let mut a = below.clone();
    a.push(i2);
    a.push(j2);
    a.extend(above.clone());
    let mut b = below;
    b.push(j2);
    b.push(i2);
    b.extend(above);
    (rangs(a), rangs(b))
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let mut ensemble: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize {
    for b in 1..=4usize {
        for c in 1..=4usize {
            for d in 1..=4usize {
                ensemble.insert(canonique((a, b, c, d)));
            }
        }
    }
}
let ordres_faibles: Vec<Tab> = ensemble.into_iter().collect();

let chambres: Vec<Tab> = ordres_faibles.iter().copied().filter(|&t| codim(t) == 0).collect();
let murs_simples: Vec<Tab> = ordres_faibles.iter().copied().filter(|&t| codim(t) == 1).collect();
let mut toutes_distinctes = true;
let mut facettes: BTreeMap<Tab, Vec<Tab>> = BTreeMap::new();
for &t in &murs_simples {
    let (a, b) = briser_le_tie(t);
    if a == b {
        toutes_distinctes = false;
    }
    facettes.entry(a).or_default().push(t);
    facettes.entry(b).or_default().push(t);
}
println!("Murs simples par cote : {}", murs_simples.len());
println!("Chaque mur touche exactement 2 chambres distinctes : {}", py_bool(toutes_distinctes));
let mut compte_facettes: BTreeMap<usize, usize> = BTreeMap::new();
for v in facettes.values() {
    *compte_facettes.entry(v.len()).or_default() += 1;
}
println!(
    "Facettes par chambre  : {:?} (3 attendu : les 3 paires de valeurs adjacentes)",
    compte_facettes
);
}


Murs simples par cote : 36


Chaque mur touche exactement 2 chambres distinctes : True


Facettes par chambre  : {3: 24} (3 attendu : les 3 paires de valeurs adjacentes)


()

**Lecture de la sortie committée** : la double incidence est **exacte**.

- **Mur → chambres** : chacun des 36 murs simples d'un côté touche **exactement 2 chambres** distinctes — les deux résolutions de son ex æquo. Une cloison n'a jamais trois pièces de chaque côté.
- **Chambre → murs** : chacune des 24 chambres a **exactement 3 facettes** — les 3 paires de valeurs *adjacentes* de son ordre. Une paire non adjacente (les valeurs 1 et 3, avec 2 entre elles) ne définit **pas** une facette : pour amener 1 et 3 à l'égalité, il faut d'abord traverser le mur où 1 rencontre 2.

C'est la structure d'**arrangement d'hyperplans** de l'espace des paiements (les murs étant les hyperplans $x_i = x_j$) — calculée ici sans en invoquer la théorie : les comptes parlent d'eux-mêmes. Au niveau du jeu complet (deux côtés), chaque chambre a donc $3 + 3 = 6$ facettes, et le nombre de murs simples du jeu vaut $2 \times 36 \times 24 = 1728$.

**Remarque de parité avec GT-21** : le théorème « le mur ne compte que s'il est habité » de GT-21 portait sur les *meilleures réponses* (quelles stratégies habitent la colonne traversée) ; ici l'habitation est *géométrique* — quelles chambres touchent le mur. Les deux lectures se complètent : GT-21 dit quand un swap **change** les équilibres, la lecture géométrique dit **où** le swap se déplace.

### 2.2 Le graphe des chambres : connexité et diamètre

Déclarer que les murs « raccordent » les chambres donne un graphe : les sommets sont les 576 chambres, une arête par mur simple traversé. La bonne nouvelle pour la grammaire des swaps : **ce graphe est connexe** — n'importe quel jeu strict est atteignable depuis n'importe quel autre en traversant des murs un à un.

In [7]:
{
// === Section 2.2 : BFS sur le graphe des chambres ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn nb_blocs(t: Tab) -> usize {
    let mut v = tab_vals(t);
    v.sort();
    v.dedup();
    v.len()
}

fn codim(t: Tab) -> usize {
    4 - nb_blocs(t)
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn voisins_nommes(g: Jeu) -> Vec<(Jeu, String)> {
    let (r, c) = g;
    let mut out: Vec<(Jeu, String)> = Vec::new();
    for k in [1usize, 2, 3] {
        out.push(((swap_valeurs_adjacentes(r, k), c), format!("R{}{}", k, k + 1)));
    }
    for k in [1usize, 2, 3] {
        out.push(((r, swap_valeurs_adjacentes(c, k)), format!("C{}{}", k, k + 1)));
    }
    out
}

fn bfs_jeu(depart: Jeu) -> BTreeMap<Jeu, usize> {
    let mut dist = BTreeMap::new();
    dist.insert(depart, 0);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for (v, _) in voisins_nommes(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let mut ensemble: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize {
    for b in 1..=4usize {
        for c in 1..=4usize {
            for d in 1..=4usize {
                ensemble.insert(canonique((a, b, c, d)));
            }
        }
    }
}
let ordres_faibles: Vec<Tab> = ensemble.into_iter().collect();

let chambres: Vec<Tab> = ordres_faibles.iter().copied().filter(|&t| codim(t) == 0).collect();

let mut adjacences: BTreeMap<Tab, Vec<Tab>> = BTreeMap::new();
for &t in &chambres {
    adjacences.insert(
        t,
        vec![
            swap_valeurs_adjacentes(t, 1),
            swap_valeurs_adjacentes(t, 2),
            swap_valeurs_adjacentes(t, 3),
        ],
    );
}
let depart: Tab = (1, 2, 3, 4);
let mut dist16: BTreeMap<Tab, usize> = BTreeMap::new();
let mut q16: VecDeque<Tab> = VecDeque::new();
dist16.insert(depart, 0);
q16.push_back(depart);
while let Some(u) = q16.pop_front() {
    let du = dist16[&u];
    for &v in &adjacences[&u] {
        if !dist16.contains_key(&v) {
            dist16.insert(v, du + 1);
            q16.push_back(v);
        }
    }
}
println!(
    "Cote 1 joueur : connexe = {} | diametre = {}",
    py_bool(dist16.len() == 24),
    dist16.values().max().unwrap()
);
let mut rep: BTreeMap<usize, usize> = BTreeMap::new();
for &d in dist16.values() {
    *rep.entry(d).or_default() += 1;
}
println!("  repartition des distances : {:?}", rep);
let dep2: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let dist2 = bfs_jeu(dep2);
println!();
println!(
    "Jeu complet : connexe = {} | diametre = {}",
    py_bool(dist2.len() == 576),
    dist2.values().max().unwrap()
);
let mut rep2: BTreeMap<usize, usize> = BTreeMap::new();
for &d in dist2.values() {
    *rep2.entry(d).or_default() += 1;
}
println!("  repartition : {:?}", rep2);
println!("  aretes = 576 x 6 / 2 = {} -- a comparer aux 1728 murs simples du 1.3", 576 * 6 / 2);
}


Cote 1 joueur : connexe = True | diametre = 6


  repartition des distances : {0: 1, 1: 3, 2: 5, 3: 6, 4: 5, 5: 3, 6: 1}


Jeu complet : connexe = True | diametre = 12


  repartition : {0: 1, 1: 6, 2: 19, 3: 42, 4: 71, 5: 96, 6: 106, 7: 96, 8: 71, 9: 42, 10: 19, 11: 6, 12: 1}


  aretes = 576 x 6 / 2 = 1728 -- a comparer aux 1728 murs simples du 1.3


()

**Lecture de la sortie committée** : le graphe des chambres est **connexe**, de diamètre **6 par côté** et **12 pour le jeu complet** — le pire cas étant l'ordre exactement renversé des deux tables ($6 + 6$ inversions).

Deux régularités lisibles dans la sortie :

- la distribution des distances d'un côté, $\{0{:}1, 1{:}3, 2{:}5, 3{:}6, 4{:}5, 5{:}3, 6{:}1\}$, est **symétrique** : le mot le plus long (l'ordre renversé) est unique, comme le plus court (l'identité) ;
- au niveau du jeu, chaque chambre a **6 voisins** (3 facettes par côté), et le compte d'arêtes $576 \times 6 / 2 = 1728$ coïncide **exactement** avec le nombre de murs simples comptés au §1.3 — chaque mur simple **est** une arête du graphe, chaque arête **est** la traversée d'un mur. Le pont entre le comptage de strates (§1) et la connectivité (§2) est un double-comptage propre.

Conséquence pour la série : le discret $\{1,2,3,4\}$ de GT-3 n'était pas un artefact de commodité — c'est un **transversal** des chambres (un point par chambre), et la connexité ci-dessus garantit que la grammaire des swaps adjacents suffit à relier tout l'univers strict.

## 3. Traverser un mur : chambre → mur → chambre

### 3.1 Un cas complet, matrices à l'appui

Prenons le **Dilemme du Prisonnier** dans l'encodage ordinal de GT-21 — Ligne $(3,1,4,2)$, Colonne $(3,4,1,2)$ — et suivons le swap de valeurs adjacentes $R(3,4)$ (Ligne échange les cases portant 3 et 4). Ce swap traverse la facette où les deux cases porteuses deviennent égales : le mur $r_{11} = r_{21}$.

In [8]:
{
// === Section 3.1 : traverser le mur r11=r21 depuis le Dilemme du Prisonnier ===

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn faire_le_tie(t: Tab, k: usize) -> Tab {
    let f = |x: usize| if x == k + 1 { k } else { x };
    canonique((f(t.0), f(t.1), f(t.2), f(t.3)))
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn briser_le_tie(t: Tab) -> (Tab, Tab) {
    let v = tab_vals(t);
    let mut sorted = v.clone();
    sorted.sort();
    let mut v_tie = 0;
    let mut i = 0;
    while i < sorted.len() {
        let mut j = i;
        while j < sorted.len() && sorted[j] == sorted[i] {
            j += 1;
        }
        if j - i == 2 {
            v_tie = sorted[i];
        }
        i = j;
    }
    let ties: Vec<usize> = (0..4).filter(|&p| v[p] == v_tie).collect();
    let (i2, j2) = (ties[0], ties[1]);
    let mut below: Vec<usize> = (0..4).filter(|&p| v[p] < v_tie).collect();
    below.sort_by_key(|&p| v[p]);
    let mut above: Vec<usize> = (0..4).filter(|&p| v[p] > v_tie).collect();
    above.sort_by_key(|&p| v[p]);
    let rangs = |ordre: Vec<usize>| -> Tab {
        let mut rk = [0usize; 4];
        for (k, &p) in ordre.iter().enumerate() {
            rk[p] = k + 1;
        }
        (rk[0], rk[1], rk[2], rk[3])
    };
    let mut a = below.clone();
    a.push(i2);
    a.push(j2);
    a.extend(above.clone());
    let mut b = below;
    b.push(j2);
    b.push(i2);
    b.extend(above);
    (rangs(a), rangs(b))
}

let afficher_jeu = |nom: &str, row: Tab, col: Tab| {
    println!("{}", nom);
    println!("  Ligne    | {:>2}  {:<2}|   Colonne | {:>2}  {:<2}|", row.0, row.1, col.0, col.1);
    println!("           | {:>2}  {:<2}|           | {:>2}  {:<2}|", row.2, row.3, col.2, col.3);
    println!();
};
let (pd_row, pd_col): (Tab, Tab) = ((3, 1, 4, 2), (3, 4, 1, 2));
afficher_jeu("CHAMBRE A -- Dilemme du Prisonnier (strict)", pd_row, pd_col);
let mur_row = faire_le_tie(pd_row, 3);
afficher_jeu("MUR (codim 1) -- Ligne indifferent r11 = r21", mur_row, pd_col);
let chambre_b_row = swap_valeurs_adjacentes(pd_row, 3);
afficher_jeu("CHAMBRE B -- apres traversee (swap R(3,4))", chambre_b_row, pd_col);
println!("Verification : mur = {:?}", mur_row);
println!("  briser_le_tie(mur) redonne les deux faces : {:?}", briser_le_tie(mur_row));
println!("  chambre A : {:?}  chambre B : {:?}", canonique(pd_row), canonique(chambre_b_row));
}


CHAMBRE A -- Dilemme du Prisonnier (strict)


  Ligne    |  3  1 |   Colonne |  3  4 |


           |  4  2 |           |  1  2 |


MUR (codim 1) -- Ligne indifferent r11 = r21


  Ligne    |  3  1 |   Colonne |  3  4 |


           |  3  2 |           |  1  2 |


... (3 outputs left out)

Verification : mur = (3, 1, 3, 2)


  briser_le_tie(mur) redonne les deux faces : ((3, 1, 4, 2), (4, 1, 3, 2))


  chambre A : (3, 1, 4, 2)  chambre B : (4, 1, 3, 2)


()

**Lecture de la sortie committée** : le triplet complet **chambre A → mur → chambre B** est exhibé avec les trois matrices — l'acceptation de l'exercice 2 est satisfaite *et démontrée* :

- **chambre A** : le PD strict, où $r_{21} = 4 > r_{11} = 3$ (Ligne gagne plus à dévier) ;
- **mur** : le jeu à égalité où $r_{11} = r_{21}$ — Ligne est **exactement indifférent** entre ses deux stratégies quand Colonne joue à gauche ; c'est le point de bascule où la défection cesse d'être strictement meilleure *en première colonne* ;
- **chambre B** : l'ordre renversé de la paire $(3,4)$ — le swap $R(3,4)$ de GT-21.

La dernière ligne vérifie mécaniquement la cohérence : le `briser_le_tie` du §2.1 appliqué au mur redonne bien **les deux** chambres — A et B — comme les deux faces de la cloison. Le swap de GT-21, redéfini géométriquement, **est** la traversée de ce mur : rien d'autre.

*Fenêtre pédagogique* : sur le mur, l'ensemble des meilleures réponses de Ligne en première colonne **grossit** (de 1 à 2 choix), puis redevient singulier de l'autre côté — avec l'autre choix. Un mur est exactement le lieu où la structure des meilleures réponses **change de cardinal** — la version locale du « mur habité » de GT-21.

### 3.2 La grammaire des swaps relue en longueurs de Coxeter

GT-21 étudiait **six** swaps par côté : $R(a,b)$ pour toutes les paires de valeurs. La géométrie les sépare en deux familles :

- $|a-b| = 1$ (valeurs **adjacentes**) : le swap traverse **un seul mur** — une facette ;
- $|a-b| \geq 2$ : les valeurs sont séparées par d'autres — le swap est un **raccourci** qui enjambe plusieurs murs.

La longueur exacte (nombre de traversées de facettes) d'une transposition de valeurs à distance $d$ est $2d-1$.

In [9]:
{
// === Section 3.2 : les swaps GT-21 en longueurs de Coxeter ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn nb_blocs(t: Tab) -> usize {
    let mut v = tab_vals(t);
    v.sort();
    v.dedup();
    v.len()
}

fn codim(t: Tab) -> usize {
    4 - nb_blocs(t)
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn appliquer_swap(t: Tab, a: usize, b: usize) -> Tab {
    let f = |v: usize| if v == a { b } else if v == b { a } else { v };
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let mut ensemble: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize {
    for b in 1..=4usize {
        for c in 1..=4usize {
            for d in 1..=4usize {
                ensemble.insert(canonique((a, b, c, d)));
            }
        }
    }
}
let ordres_faibles: Vec<Tab> = ensemble.into_iter().collect();

let chambres: Vec<Tab> = ordres_faibles.iter().copied().filter(|&t| codim(t) == 0).collect();

let mut adjacences: BTreeMap<Tab, Vec<Tab>> = BTreeMap::new();
for &t in &chambres {
    adjacences.insert(
        t,
        vec![
            swap_valeurs_adjacentes(t, 1),
            swap_valeurs_adjacentes(t, 2),
            swap_valeurs_adjacentes(t, 3),
        ],
    );
}

let distance_cote = |u0: Tab, cible: Tab| -> usize {
    let mut d: BTreeMap<Tab, usize> = BTreeMap::new();
    let mut file: VecDeque<Tab> = VecDeque::new();
    d.insert(u0, 0);
    file.push_back(u0);
    while let Some(x) = file.pop_front() {
        if x == cible {
            return d[&x];
        }
        let dx = d[&x];
        for &y in &adjacences[&x] {
            if !d.contains_key(&y) {
                d.insert(y, dx + 1);
                file.push_back(y);
            }
        }
    }
    0 // inatteignable : le graphe est connexe (mesure cellule 16)
};
println!("Swap    | murs traverses (mesure BFS) | formule 2d-1 | statut");
println!("{}", "-".repeat(74));
let mut longueur_unique = true;
let mut mesures: BTreeMap<(usize, usize), usize> = BTreeMap::new();
for (a, b) in [(1usize, 2usize), (2, 3), (3, 4), (1, 3), (2, 4), (1, 4)] {
    let d = if a > b { a - b } else { b - a };
    let mut trouvees: BTreeSet<usize> = BTreeSet::new();
    for &t in &chambres {
        trouvees.insert(distance_cote(t, appliquer_swap(t, a, b)));
    }
    longueur_unique = longueur_unique && trouvees.len() == 1;
    mesures.insert((a, b), *trouvees.iter().min().unwrap());
    let statut = if d == 1 { "facette (1 mur)" } else { "raccourci" };
    println!(
        "R({},{})  |            {:2}              |      {:2}      |  {}",
        a,
        b,
        mesures[&(a, b)],
        2 * d - 1,
        statut
    );
}
println!();
println!(
    "Une seule longueur par swap, quelle que soit la chambre de depart : {}",
    py_bool(longueur_unique)
);
let confirme = mesures
    .iter()
    .all(|(&(a, b), &m)| m == 2 * (if a > b { a - b } else { b - a }) - 1);
println!("La mesure confirme la formule 2d-1 sur les six swaps : {}", py_bool(confirme));
let t0 = (1usize, 2, 3, 4);
let direct = appliquer_swap(t0, 1, 3);
let compose = appliquer_swap(appliquer_swap(appliquer_swap(t0, 1, 2), 2, 3), 1, 2);
println!();
println!("R(1,3) applique a (1,2,3,4)      : {:?}", direct);
println!("R(1,2) puis R(2,3) puis R(1,2)   : {:?}", compose);
println!("Egalite (le raccourci = 3 facettes) : {}", py_bool(direct == compose));
}


Swap    | murs traverses (mesure BFS) | formule 2d-1 | statut


--------------------------------------------------------------------------


R(1,2)  |             1              |       1      |  facette (1 mur)


R(2,3)  |             1              |       1      |  facette (1 mur)


R(3,4)  |             1              |       1      |  facette (1 mur)


R(1,3)  |             3              |       3      |  raccourci


... (4 outputs left out)

R(1,3) applique a (1,2,3,4)      : (3, 2, 1, 4)


R(1,2) puis R(2,3) puis R(1,2)   : (3, 2, 1, 4)


Egalite (le raccourci = 3 facettes) : True


()

**Lecture de la sortie committée** : la grammaire de GT-21 se recompose géométriquement.

La colonne du milieu **mesure** — c'est la distance dans le graphe des chambres du §2.2, parcourue en largeur, et elle ne dépend pas de la chambre de départ ; la colonne de droite **prédit** par la formule $2d-1$. Que les deux coïncident sur les six swaps est le résultat, et non une tautologie : une colonne calculée par la formule qu'elle illustre ne prouverait rien.

- les trois swaps **adjacents** $R(1,2)$, $R(2,3)$, $R(3,4)$ sont les **générateurs** : chacun traverse exactement un mur (une facette), et la mesure le confirme. Avec leurs homologues Colonne, ils engendrent tout le graphe des 576 chambres (§2.2) ;
- les trois **raccourcis** $R(1,3)$, $R(2,4)$, $R(1,4)$ enjambent respectivement 3, 3 et 5 murs — **mesurés** à la même distance, pas déduits de la formule. La décomposition est vérifiée dans la sortie : $R(1,3) = R(1,2) \circ R(2,3) \circ R(1,2)$ — le raccourci et la cascade de facettes aboutissent au **même** 4-uplet, mais la cascade *rencontre* les murs intermédiaires (des jeux à égalités où la structure des meilleures réponses peut basculer), quand le raccourci les saute.

C'est la réponse géométrique à une question laissée ouverte par GT-21 : pourquoi certains swaps « ne comptent-ils » pas (préservation des équilibres) ? Parce qu'un swap n'agit sur les équilibres qu'à travers les murs qu'il traverse qui sont **effectivement habités** (au sens meilleures réponses de GT-21). Les facettes rencontrées par la cascade sont autant d'occasions de bascule ; le raccourci n'échantillonne que l'état final.

## 4. Make tie / break tie : les deux opérations de Bruns-Kimmich

### 4.1 Les opérations, implémentées

La description géométrique s'accompagne de deux opérations duuales (rapportées de Bruns-Kimmich, implémentées ici) :

- **make tie** : depuis une chambre, rapprocher deux valeurs adjacentes jusqu'à l'égalité — on **pose** le jeu sur un mur (codim +1) ;
- **break tie** : depuis un mur, séparer l'ex æquo dans un sens ou dans l'autre — on **retombe** dans l'une des deux chambres adjacentes (codim −1).

Ce sont exactement les briques des §2-3, promues au rang d'API.

In [10]:
{
// === Section 4.1 : make_tie / break_tie, operations duales ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn nb_blocs(t: Tab) -> usize {
    let mut v = tab_vals(t);
    v.sort();
    v.dedup();
    v.len()
}

fn codim(t: Tab) -> usize {
    4 - nb_blocs(t)
}

fn faire_le_tie(t: Tab, k: usize) -> Tab {
    let f = |x: usize| if x == k + 1 { k } else { x };
    canonique((f(t.0), f(t.1), f(t.2), f(t.3)))
}

fn briser_le_tie(t: Tab) -> (Tab, Tab) {
    let v = tab_vals(t);
    let mut sorted = v.clone();
    sorted.sort();
    let mut v_tie = 0;
    let mut i = 0;
    while i < sorted.len() {
        let mut j = i;
        while j < sorted.len() && sorted[j] == sorted[i] {
            j += 1;
        }
        if j - i == 2 {
            v_tie = sorted[i];
        }
        i = j;
    }
    let ties: Vec<usize> = (0..4).filter(|&p| v[p] == v_tie).collect();
    let (i2, j2) = (ties[0], ties[1]);
    let mut below: Vec<usize> = (0..4).filter(|&p| v[p] < v_tie).collect();
    below.sort_by_key(|&p| v[p]);
    let mut above: Vec<usize> = (0..4).filter(|&p| v[p] > v_tie).collect();
    above.sort_by_key(|&p| v[p]);
    let rangs = |ordre: Vec<usize>| -> Tab {
        let mut rk = [0usize; 4];
        for (k, &p) in ordre.iter().enumerate() {
            rk[p] = k + 1;
        }
        (rk[0], rk[1], rk[2], rk[3])
    };
    let mut a = below.clone();
    a.push(i2);
    a.push(j2);
    a.extend(above.clone());
    let mut b = below;
    b.push(j2);
    b.push(i2);
    b.extend(above);
    (rangs(a), rangs(b))
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let mut ensemble: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize {
    for b in 1..=4usize {
        for c in 1..=4usize {
            for d in 1..=4usize {
                ensemble.insert(canonique((a, b, c, d)));
            }
        }
    }
}
let ordres_faibles: Vec<Tab> = ensemble.into_iter().collect();

let murs_simples: Vec<Tab> = ordres_faibles.iter().copied().filter(|&t| codim(t) == 1).collect();

let make_tie = |jeu: Jeu, cote: bool, k: usize| -> Jeu {
    // cote = true : ligne
    if cote {
        (faire_le_tie(jeu.0, k), jeu.1)
    } else {
        (jeu.0, faire_le_tie(jeu.1, k))
    }
};
let break_tie = |jeu: Jeu, cote: bool, bas: bool| -> Jeu {
    let t = if cote { jeu.0 } else { jeu.1 };
    let v = tab_vals(t);
    let mut sorted = v.clone();
    sorted.sort();
    let mut v_tie: Option<usize> = None;
    let mut i = 0;
    while i < sorted.len() {
        let mut j = i;
        while j < sorted.len() && sorted[j] == sorted[i] {
            j += 1;
        }
        if j - i == 2 {
            v_tie = Some(sorted[i]);
        }
        i = j;
    }
    let v0 = match v_tie {
        None => return jeu, // pas d'ex aequo : operation identite
        Some(x) => x,
    };
    let mut below: Vec<usize> = (0..4).filter(|&p| v[p] < v0).collect();
    below.sort_by_key(|&p| v[p]);
    let mut above: Vec<usize> = (0..4).filter(|&p| v[p] > v0).collect();
    above.sort_by_key(|&p| v[p]);
    let ties: Vec<usize> = (0..4).filter(|&p| v[p] == v0).collect();
    let (i2, j2) = (ties[0], ties[1]);
    let mut ordre = below.clone();
    if bas {
        ordre.push(j2);
        ordre.push(i2);
    } else {
        ordre.push(i2);
        ordre.push(j2);
    }
    ordre.extend(above);
    let mut rk = [0usize; 4];
    for (k, &p) in ordre.iter().enumerate() {
        rk[p] = k + 1;
    }
    let t2 = (rk[0], rk[1], rk[2], rk[3]);
    if cote {
        (t2, jeu.1)
    } else {
        (jeu.0, t2)
    }
};
let pd: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let mur = make_tie(pd, true, 3);
let retour_haut = break_tie(mur, true, false);
let retour_bas = break_tie(mur, true, true);
println!("Chambre (PD)        : {:?}", pd);
println!(
    "make_tie(ligne, 3)  : {:?}  (codim {} )",
    mur,
    codim(mur.0) + codim(mur.1)
);
println!(
    "break_tie(haut)     : {:?} -> retrouve le PD : {}",
    retour_haut,
    py_bool(retour_haut == pd)
);
println!("break_tie(bas)      : {:?} -> l'autre chambre (chambre B du 3.1)", retour_bas);
println!();
println!("Le mur porte bien 2 sorties distinctes : {}", py_bool(retour_haut != retour_bas));
let mut deux_chambres = true;
let mut memes_faces = true;
for &t in &murs_simples {
    let jeu_mur: Jeu = (t, (1, 2, 3, 4));
    let haut = break_tie(jeu_mur, true, false).0;
    let bas = break_tie(jeu_mur, true, true).0;
    deux_chambres = deux_chambres && nb_blocs(haut) == 4 && nb_blocs(bas) == 4 && haut != bas;
    memes_faces = memes_faces && (haut, bas) == briser_le_tie(t);
}
println!();
println!("Sur les 36 murs simples : deux chambres strictes distinctes = {}", py_bool(deux_chambres));
println!("Sur les 36 murs simples : les deux faces de briser_le_tie  = {}", py_bool(memes_faces));
}


Chambre (PD)        : ((3, 1, 4, 2), (3, 4, 1, 2))


make_tie(ligne, 3)  : ((3, 1, 3, 2), (3, 4, 1, 2))  (codim 1 )


break_tie(haut)     : ((3, 1, 4, 2), (3, 4, 1, 2)) -> retrouve le PD : True


break_tie(bas)      : ((4, 1, 3, 2), (3, 4, 1, 2)) -> l'autre chambre (chambre B du 3.1)


Le mur porte bien 2 sorties distinctes : True


Sur les 36 murs simples : deux chambres strictes distinctes = True


Sur les 36 murs simples : les deux faces de briser_le_tie  = True


()

**Lecture de la sortie committée** : l'aller-retour est **mécaniquement vérifié**. `make_tie` fait perdre une dimension (le jeu quitte l'ouvert des chambres pour la cloison), `break_tie` la rend — dans **un des deux sens** : la sortie `haut` retrouve le PD d'origine, la sortie `bas` atterrit dans la chambre B du §3.1. Les deux sorties sont distinctes : c'est la matérialisation calculée du « plusieurs prolongements sont possibles » de la thèse botanique.

Le cas du PD ne vaut pas preuve à lui seul — c'est même le piège de cette opération : sur un mur dont la valeur séparée n'a pas de voisine immédiate déjà placée, séparer « sur place » suffit ; sur les autres, cela fabrique une nouvelle égalité et l'on reste sur un mur. La sortie le tranche sur **les 36 murs simples d'un côté** : chacun rend deux chambres strictes distinctes, et ce sont exactement les deux faces que `briser_le_tie` (§2.1) leur connaissait déjà. L'opération du §4.1 et celle du §2.1 sont donc la même, vue une fois depuis le mur et une fois depuis la chambre.

Un jeu à égalités n'est donc **pas** une spécification incomplète d'un jeu strict : c'est un objet dont le **type** même est différent — un point de l'architecture, avec ses deux faces — et dont le prolongement exige un choix supplémentaire (un sens). Toute la différence entre *assouplir une définition* et *habiter le bord*.

### 4.2 Trois archétypes au bout des murs

Bruns-Kimmich (rapporté) décrivent les jeux 2×2 primitifs comme engendrés depuis des murs par *make/break tie*, sous trois archétypes : **indépendance**, **coordination**, **échange**. Nos implémentations permettent d'en donner des **instanciations calculées** — la définition exacte des archétypes dans la source relève de la dette de sources (§Sources) ; ce qui suit instancie chaque nom par un mur de géométrie naturelle, obtenu par `make_tie` et vérifié par `break_tie`.

In [11]:
{
// === Section 4.2 : trois instanciations (RAPPORTEES, cf. Sources) ===

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn faire_le_tie(t: Tab, k: usize) -> Tab {
    let f = |x: usize| if x == k + 1 { k } else { x };
    canonique((f(t.0), f(t.1), f(t.2), f(t.3)))
}

fn nb_blocs(t: Tab) -> usize {
    let mut v = tab_vals(t);
    v.sort();
    v.dedup();
    v.len()
}

fn codim(t: Tab) -> usize {
    4 - nb_blocs(t)
}

let make_tie = |jeu: Jeu, cote: bool, k: usize| -> Jeu {
    // cote = true : ligne
    if cote {
        (faire_le_tie(jeu.0, k), jeu.1)
    } else {
        (jeu.0, faire_le_tie(jeu.1, k))
    }
};

println!("Archetype       | mur obtenu (Ligne | Colonne)            | geometrie du tie");
println!("{}", "-".repeat(90));
let base_ind: Jeu = ((1, 3, 2, 4), (1, 3, 2, 4));
let mur_ind = make_tie(base_ind, true, 1);
println!(
    "Independance    | {:18} | {:18} | vertical (r11 = r21)",
    format!("{:?}", mur_ind.0),
    format!("{:?}", mur_ind.1)
);
let base_co: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let mur_co = make_tie(make_tie(base_co, true, 2), false, 2);
println!(
    "Coordination    | {:18} | {:18} | anti-diagonal x2, codim {}",
    format!("{:?}", mur_co.0),
    format!("{:?}", mur_co.1),
    codim(mur_co.0) + codim(mur_co.1)
);
let base_ech: Jeu = ((1, 2, 3, 4), (2, 1, 4, 3));
let mur_ech = make_tie(base_ech, true, 1);
println!(
    "Echange         | {:18} | {:18} | horizontal (r11 = r12)",
    format!("{:?}", mur_ech.0),
    format!("{:?}", mur_ech.1)
);
println!();
println!("Chaque mur se brise dans 2 sens -> chaque archetype borde 2 familles de jeux stricts.");
println!("(Instanciations de notre cru ; la definition source des archetypes : dette, cf. Sources.)");
}


Archetype       | mur obtenu (Ligne | Colonne)            | geometrie du tie


------------------------------------------------------------------------------------------


Independance    | (1, 2, 1, 3)       | (1, 3, 2, 4)       | vertical (r11 = r21)


Coordination    | (1, 2, 2, 3)       | (1, 2, 2, 3)       | anti-diagonal x2, codim 2


Echange         | (1, 1, 2, 3)       | (2, 1, 4, 3)       | horizontal (r11 = r12)


Chaque mur se brise dans 2 sens -> chaque archetype borde 2 familles de jeux stricts.


(Instanciations de notre cru ; la definition source des archetypes : dette, cf. Sources.)


()

**Lecture de la sortie committée** : chaque archétype est matérialisé par un jeu mur obtenu **par calcul** depuis un jeu strict :

- **indépendance** — mur *vertical* de Ligne ($r_{11} = r_{21}$) : Ligne exactement indifférent entre ses deux stratégies face à la colonne gauche ; brisé vers le haut ou le bas, ce mur fabrique les jeux où Ligne a une préférence stricte — c'est le voisinage des décisions *unilatérales* ;
- **coordination** — double mur *anti-diagonal* (codim 2, une **arête** de l'architecture) : les deux joueurs lisent leur duel sur la même paire de cases ; brisé dans le même sens des deux côtés → coordination pure, en sens opposés → conflit — le voisinage des jeux de rencontre ;
- **échange** — mur *horizontal* de Ligne ($r_{11} = r_{12}$) : le gain de Ligne en haut ne dépend pas de l'action de Colonne ; brisé, il fabrique les jeux où l'issue dépend du croisement des deux décisions — le voisinage des marchandages.

Ces instanciations **calculées** bornent honnêtement ce que ce notebook établit : les opérations, les comptes, les incidences. La généalogie complète des 2×2 par archétypes — les trois primitives de Bruns-Kimmich — est **rapportée**, non dérivée : elle exige la source primaire (§Sources, dette n°4).

## 5. L'espace des 576 jeux et ses générateurs

La première moitié a décrit l'espace et ses strates ; celle-ci pose la question qui reste : *à quelle distance de swap sont deux jeux, et par quels chemins les relie-t-on ?* La distance se mesure en murs traversés — ce que la §2.2 appelait une facette, et que la §3.1 traversait sur un cas complet.

Deux questions d'analyse d'algorithme s'y ajoutent, qui structurent toute cette moitié :

1. **Le générateur propose** — un parcours en largeur (BFS) explore l'espace niveau par niveau et produit un plus court chemin candidat.
2. **Le certificat garantit** — un module compagnon en Lean (`game_theory_lean/Swaps/Basic.lean`) certifie qu'un chemin proposé est bien formé et mène bien du départ à l'arrivée ; sur le cas témoin Dilemme → Chicken, il certifie aussi qu'aucun chemin plus court n'existe. Le générateur Python et le vérificateur Lean restent deux artefacts distincts, chacun à sa place.

Toutes les mesures portent sur l'espace complet des 576 jeux stricts — le quotient par renommage des stratégies (144 classes) est une dette rapportée, jamais affirmée ici.


In [12]:
{
// Convention GameTheory-3 / GameTheory-21 : une table porte les rangs 1-4

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn canonique(t: Tab) -> Tab {
    let mut vals = tab_vals(t);
    vals.sort();
    vals.dedup();
    let f = |v: usize| vals.iter().position(|&x| x == v).unwrap() + 1;
    (f(t.0), f(t.1), f(t.2), f(t.3))
}

fn nb_blocs(t: Tab) -> usize {
    let mut v = tab_vals(t);
    v.sort();
    v.dedup();
    v.len()
}

fn codim(t: Tab) -> usize {
    4 - nb_blocs(t)
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let mut ensemble: BTreeSet<Tab> = BTreeSet::new();
for a in 1..=4usize {
    for b in 1..=4usize {
        for c in 1..=4usize {
            for d in 1..=4usize {
                ensemble.insert(canonique((a, b, c, d)));
            }
        }
    }
}
let ordres_faibles: Vec<Tab> = ensemble.into_iter().collect();

let chambres: Vec<Tab> = ordres_faibles.iter().copied().filter(|&t| codim(t) == 0).collect();

let mut permset: BTreeSet<Tab> = BTreeSet::new();
let mut perm: Vec<usize> = vec![1, 2, 3, 4];
permset.insert((1, 2, 3, 4));
//Heap
fn heap(k: usize, a: &mut Vec<usize>, out: &mut BTreeSet<Tab>) {
    if k == 1 {
        out.insert((a[0], a[1], a[2], a[3]));
        return;
    }
    for i in 0..k {
        heap(k - 1, a, out);
        if k % 2 == 0 {
            a.swap(i, k - 1);
        } else {
            a.swap(0, k - 1);
        }
    }
}
heap(4, &mut perm, &mut permset);
let perms: Vec<Tab> = permset.into_iter().collect(); // trie lexicographique = ordre itertools
let mut jeux31: Vec<Jeu> = Vec::new();
for &r in &perms {
    for &c in &perms {
        jeux31.push((r, c));
    }
}
// Python : swap_adjacent = swap_valeurs_adjacentes ; « is » -> True PAR CONSTRUCTION
// (deux noms, un seul objet fonction). En Rust : une seule definition, l'alias
// delegue ; l'identite par construction se rend par l'egalite comportementale
// exhaustive sur le domaine (24 tables x 3 rangs) — la comparaison de pointeurs
// de fonctions est interdite par le lint `unpredictable_function_pointer_comparisons`.
let swap_adjacent = |t: Tab, k: usize| -> Tab { swap_valeurs_adjacentes(t, k) };
let voisins = |g: Jeu| -> Vec<Jeu> {
    let (r, c) = g;
    let mut out: Vec<Jeu> = Vec::new();
    for k in [1usize, 2, 3] {
        out.push((swap_valeurs_adjacentes(r, k), c));
    }
    for k in [1usize, 2, 3] {
        out.push((r, swap_valeurs_adjacentes(c, k)));
    }
    out
};
let classic: Vec<(&str, Jeu)> = vec![
    ("Dilemme", ((3, 1, 4, 2), (3, 4, 1, 2))),
    ("Chicken", ((3, 2, 4, 1), (3, 4, 2, 1))),
    ("Deadlock", ((2, 1, 4, 3), (2, 4, 1, 3))),
    ("Chasse au cerf", ((4, 1, 3, 2), (4, 3, 1, 2))),
    ("Harmonie", ((4, 3, 2, 1), (4, 2, 3, 1))),
    ("Pennies", ((4, 1, 2, 3), (1, 4, 3, 2))),
];
let cherche = |nom: &str| -> Jeu { classic.iter().find(|(n, _)| *n == nom).unwrap().1 };
println!("Espace : {} jeux stricts, {} generateurs par jeu", jeux31.len(), voisins(jeux31[0]).len());
// Identite par construction : swap_adjacent est un alias de la meme definition.
// Le domaine de l'alias est celui de la cellule 16 (24 ordres stricts x 3 rangs),
// pas les 256 4-uplets : hors permutation de {1,2,3,4} le swap n'est pas defini.
let generateur_partage = chambres
    .iter()
    .all(|&t| (1..=3usize).all(|k| swap_adjacent(t, k) == swap_valeurs_adjacentes(t, k)));
println!("generateur partage avec le 2.2 : {}", py_bool(generateur_partage));
}


Espace : 576 jeux stricts, 6 generateurs par jeu


generateur partage avec le 2.2 : True


()

## 6. La distance d'un jeu à l'autre

La **distance de swap** entre deux jeux est la longueur du plus court chemin de générateurs qui relie l'un à l'autre — le nombre minimal de murs traversés, au sens du §3.1. C'est la distance de graphe sur les 576 sommets, et le parcours en largeur la mesure exactement : le BFS découvre les jeux par sphères croissantes, et la sphère où un jeu apparaît pour la première fois est sa distance au point de départ.

Lançons le BFS depuis le Dilemme du prisonnier — le jeu le plus commenté de la théorie — et demandons les distances aux cinq autres archétypes canoniques.

In [13]:
{
// === Section 6 : BFS depuis le Dilemme, avec parents pour reconstruire les chemins ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn voisins_nommes(g: Jeu) -> Vec<(Jeu, String)> {
    let (r, c) = g;
    let mut out: Vec<(Jeu, String)> = Vec::new();
    for k in [1usize, 2, 3] {
        out.push(((swap_valeurs_adjacentes(r, k), c), format!("R{}{}", k, k + 1)));
    }
    for k in [1usize, 2, 3] {
        out.push(((r, swap_valeurs_adjacentes(c, k)), format!("C{}{}", k, k + 1)));
    }
    out
}

fn bfs_jeu_parent(depart: Jeu) -> (BTreeMap<Jeu, usize>, BTreeMap<Jeu, Option<Jeu>>) {
    // Ordre d'exploration Python : R1,R2,R3,C1,C2,C3, une seule boucle sur les six.
    // (le « voisins » non-nomme de la cellule 31 a les memes trois-blocs separes ;
    // l'ordre R-puis-C est inchange : les chemins Pennies/Harmonie du temoin
    // dependent de cet ordre inter-cote.)
    let mut dist = BTreeMap::new();
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    dist.insert(depart, 0);
    parent.insert(depart, None);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for (v, _) in voisins_nommes(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                parent.insert(v, Some(u));
                q.push_back(v);
            }
        }
    }
    (dist, parent)
}

let classic: Vec<(&str, Jeu)> = vec![
    ("Dilemme", ((3, 1, 4, 2), (3, 4, 1, 2))),
    ("Chicken", ((3, 2, 4, 1), (3, 4, 2, 1))),
    ("Deadlock", ((2, 1, 4, 3), (2, 4, 1, 3))),
    ("Chasse au cerf", ((4, 1, 3, 2), (4, 3, 1, 2))),
    ("Harmonie", ((4, 3, 2, 1), (4, 2, 3, 1))),
    ("Pennies", ((4, 1, 2, 3), (1, 4, 3, 2))),
];
let cherche = |nom: &str| -> Jeu { classic.iter().find(|(n, _)| *n == nom).unwrap().1 };

let (dist_pd, parent_pd) = bfs_jeu_parent(cherche("Dilemme"));

let (dist_pd, parent_pd) = bfs_jeu_parent(cherche("Dilemme"));
println!("jeux atteints depuis le Dilemme : {} / 576", dist_pd.len());
println!("excentricité du Dilemme : {}", dist_pd.values().max().unwrap());
println!();
println!("Distances depuis le Dilemme :");
for (nom, g) in &classic {
    println!("  Dilemme -> {:16} distance = {}", nom, dist_pd[g]);
}
let _ = ();
}


jeux atteints depuis le Dilemme : 576 / 576


excentricité du Dilemme : 12


Distances depuis le Dilemme :


  Dilemme -> Dilemme          distance = 0


  Dilemme -> Chicken          distance = 2


  Dilemme -> Deadlock         distance = 2


  Dilemme -> Chasse au cerf   distance = 2


  Dilemme -> Harmonie         distance = 6


  Dilemme -> Pennies          distance = 5


()

### Interprétation : trois cousins à distance 2, un antagoniste à distance 12

Trois archétypes sont à distance 2 du Dilemme — **Chicken, Deadlock et la chasse au cerf** — et un seul est à distance 12 : son **complément ordinal**, le jeu où chaque rang $r$ est remplacé par $5 - r$ chez les deux joueurs (l'« anti-Dilemme », unique point de la sphère maximale). Entre les deux, Pennies à 5 et Harmonie à 6.

La distance 2 des trois cousins n'est pas un artefact d'encodage : chacun s'obtient du Dilemme en échangeant, **chez chaque joueur**, une paire différente de rangs adjacents :

- **Chicken** : l'ordre des deux *pires* rangs ($P \leftrightarrow S$) — c'est tout ce qui sépare la coopération forcée du Dilemme de l'intimidation réciproque ;
- **Deadlock** : l'ordre des rangs du *milieu* ;
- **la chasse au cerf** : l'ordre des deux *meilleurs* rangs ($T \leftrightarrow R$) — le pari de confiance.

Reconstruisons ces chemins avec la table des parents du BFS.

In [14]:
{
// === Reconstruction des chemins : nommer chaque etape, remonter les parents ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn voisins_nommes(g: Jeu) -> Vec<(Jeu, String)> {
    let (r, c) = g;
    let mut out: Vec<(Jeu, String)> = Vec::new();
    for k in [1usize, 2, 3] {
        out.push(((swap_valeurs_adjacentes(r, k), c), format!("R{}{}", k, k + 1)));
    }
    for k in [1usize, 2, 3] {
        out.push(((r, swap_valeurs_adjacentes(c, k)), format!("C{}{}", k, k + 1)));
    }
    out
}

fn bfs_jeu_parent(depart: Jeu) -> (BTreeMap<Jeu, usize>, BTreeMap<Jeu, Option<Jeu>>) {
    // Ordre d'exploration Python : R1,R2,R3,C1,C2,C3, une seule boucle sur les six.
    // (le « voisins » non-nomme de la cellule 31 a les memes trois-blocs separes ;
    // l'ordre R-puis-C est inchange : les chemins Pennies/Harmonie du temoin
    // dependent de cet ordre inter-cote.)
    let mut dist = BTreeMap::new();
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    dist.insert(depart, 0);
    parent.insert(depart, None);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for (v, _) in voisins_nommes(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                parent.insert(v, Some(u));
                q.push_back(v);
            }
        }
    }
    (dist, parent)
}

let classic: Vec<(&str, Jeu)> = vec![
    ("Dilemme", ((3, 1, 4, 2), (3, 4, 1, 2))),
    ("Chicken", ((3, 2, 4, 1), (3, 4, 2, 1))),
    ("Deadlock", ((2, 1, 4, 3), (2, 4, 1, 3))),
    ("Chasse au cerf", ((4, 1, 3, 2), (4, 3, 1, 2))),
    ("Harmonie", ((4, 3, 2, 1), (4, 2, 3, 1))),
    ("Pennies", ((4, 1, 2, 3), (1, 4, 3, 2))),
];
let cherche = |nom: &str| -> Jeu { classic.iter().find(|(n, _)| *n == nom).unwrap().1 };

let (dist_pd, parent_pd) = bfs_jeu_parent(cherche("Dilemme"));

let nommer_etape = |g1: Jeu, g2: Jeu| -> String {
    let (r1, c1) = g1;
    let (r2, c2) = g2;
    for k in [1usize, 2, 3] {
        if swap_valeurs_adjacentes(r1, k) == r2 {
            return format!("R{}{}", k, k + 1);
        }
        if swap_valeurs_adjacentes(c1, k) == c2 {
            return format!("C{}{}", k, k + 1);
        }
    }
    String::new() // inatteignable : g1 et g2 voisins (assert de la source)
};
let reconstruire = |arrivee: Jeu| -> Vec<String> {
    let mut chemin: Vec<String> = Vec::new();
    let mut v = arrivee;
    while let Some(Some(p)) = parent_pd.get(&v) {
        chemin.push(nommer_etape(*p, v));
        v = *p;
    }
    chemin.reverse();
    chemin
};
let complement = |g: Jeu| -> Jeu {
    let f = |t: Tab| (5 - t.0, 5 - t.1, 5 - t.2, 5 - t.3);
    (f(g.0), f(g.1))
};
for nom in ["Chicken", "Deadlock", "Chasse au cerf", "Pennies", "Harmonie"] {
    let g = cherche(nom);
    let chemin = reconstruire(g);
    println!("Dilemme -> {:16} (distance {:2}) : {}", nom, dist_pd[&g], chemin.join(" "));
}
let anti = complement(cherche("Dilemme"));
let n12 = dist_pd.values().filter(|&&d| d == 12).count();
println!("\nanti-Dilemme = {:?}", anti);
println!("distance = {} ; jeux à cette distance : {}", dist_pd[&anti], n12);
}


Dilemme -> Chicken          (distance  2) : R12 C12


Dilemme -> Deadlock         (distance  2) : R23 C23


Dilemme -> Chasse au cerf   (distance  2) : R34 C34


Dilemme -> Pennies          (distance  5) : R34 R23 C12 C23 C12


Dilemme -> Harmonie         (distance  6) : R12 R34 R23 C12 C34 C23


anti-Dilemme = ((2, 4, 1, 3), (2, 1, 4, 3))


distance = 12 ; jeux à cette distance : 1


()

### Interprétation : la géographie du Dilemme

Le Dilemme habite un carrefour : ses trois cousins célèbres sont à deux pas, chacun dans une direction cardinale différente (bas, milieu, haut de l'échelle des rangs), et l'unique jeu le plus lointain — à 12 swaps — est son **complément ordinal exact**, celui où chaque cellule qui était un bon rang devient un mauvais. Dans ce jeu antagoniste, la situation du Dilemme est intégralement inversée : ce que la coopération produisait de mieux devient pire. La mesure dit autrement la même chose que le §2.2 : traverser les 12 murs qui séparent un jeu de son complément, c'est renverser toute son échelle de valeurs — et aucun chemin ne peut être plus long, car changer les deux tables complètes exige au plus 6 swaps adjacents chacune (le retournement d'un ordre de 4 éléments).

Le point mérite d'être regardé de plus près : la structure de toutes ces distances est remarquablement régulière, et c'est l'objet du §7.

## 7. La distribution des distances : des sphères emboîtées

Le BFS donne la distance de *chaque* jeu au Dilemme — autrement dit la cardinalité de chaque sphère $S_d = \{G : d(\text{Dilemme}, G) = d\}$ pour $d$ de 0 à 12. Si la géographie du Dilemme était quelconque, ces tailles n'auraient aucune raison d'être régulières. Mesurons.

In [15]:
{
// === Section 7 : distribution des distances depuis le Dilemme ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn voisins_nommes(g: Jeu) -> Vec<(Jeu, String)> {
    let (r, c) = g;
    let mut out: Vec<(Jeu, String)> = Vec::new();
    for k in [1usize, 2, 3] {
        out.push(((swap_valeurs_adjacentes(r, k), c), format!("R{}{}", k, k + 1)));
    }
    for k in [1usize, 2, 3] {
        out.push(((r, swap_valeurs_adjacentes(c, k)), format!("C{}{}", k, k + 1)));
    }
    out
}

fn bfs_jeu_parent(depart: Jeu) -> (BTreeMap<Jeu, usize>, BTreeMap<Jeu, Option<Jeu>>) {
    // Ordre d'exploration Python : R1,R2,R3,C1,C2,C3, une seule boucle sur les six.
    // (le « voisins » non-nomme de la cellule 31 a les memes trois-blocs separes ;
    // l'ordre R-puis-C est inchange : les chemins Pennies/Harmonie du temoin
    // dependent de cet ordre inter-cote.)
    let mut dist = BTreeMap::new();
    let mut parent: BTreeMap<Jeu, Option<Jeu>> = BTreeMap::new();
    dist.insert(depart, 0);
    parent.insert(depart, None);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for (v, _) in voisins_nommes(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                parent.insert(v, Some(u));
                q.push_back(v);
            }
        }
    }
    (dist, parent)
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let classic: Vec<(&str, Jeu)> = vec![
    ("Dilemme", ((3, 1, 4, 2), (3, 4, 1, 2))),
    ("Chicken", ((3, 2, 4, 1), (3, 4, 2, 1))),
    ("Deadlock", ((2, 1, 4, 3), (2, 4, 1, 3))),
    ("Chasse au cerf", ((4, 1, 3, 2), (4, 3, 1, 2))),
    ("Harmonie", ((4, 3, 2, 1), (4, 2, 3, 1))),
    ("Pennies", ((4, 1, 2, 3), (1, 4, 3, 2))),
];
let cherche = |nom: &str| -> Jeu { classic.iter().find(|(n, _)| *n == nom).unwrap().1 };

let (dist_pd, parent_pd) = bfs_jeu_parent(cherche("Dilemme"));

let mut distribution: BTreeMap<usize, usize> = BTreeMap::new();
for &d in dist_pd.values() {
    *distribution.entry(d).or_default() += 1;
}
let total38: usize = distribution.values().sum();
let ds: Vec<String> = (0..13).map(|d| format!("{:6}", d)).collect();
let ns: Vec<String> = (0..13).map(|d| format!("{:6}", distribution[&d])).collect();
println!("d : {}", ds.join(" "));
println!("N : {}", ns.join(" "));
println!("total : {}", total38);
println!();
for d in 0..13usize {
    println!("  d = {:2} | {:48} {:4}", d, "#".repeat(distribution[&d] / 2), distribution[&d]);
}
let symetrie = (0..13).all(|d| distribution[&d] == distribution[&(12 - d)]);
println!("\nsymétrie d <-> 12 - d : {}", py_bool(symetrie));
println!("maximum atteint en d = 6 : {} jeux", distribution[&6]);
}


d :      0      1      2      3      4      5      6      7      8      9     10     11     12


N :      1      6     19     42     71     96    106     96     71     42     19      6      1


total : 576


  d =  0 |                                                     1


  d =  1 | ###                                                 6


  d =  2 | #########                                          19


... (9 outputs left out)

  d = 12 |                                                     1


symétrie d <-> 12 - d : True


maximum atteint en d = 6 : 106 jeux


()

### Interprétation : une distribution symétrique et factorisée

La distribution est **parfaitement symétrique** autour de 6, en cloche, avec un unique jeu aux deux extrémités ($d = 0$ le Dilemme lui-même, $d = 12$ son complément) et un plateau de 106 jeux à distance 6. Cette régularité appelle une explication structurelle — et elle en a une, calculable de façon indépendante.

La distribution des longueurs de mots sur les 24 permutations d'un côté n'est autre que la distribution des **nombres d'inversions** de $S_4$ : $\{1, 3, 5, 6, 5, 3, 1\}$ — la cellule suivante la mesure, elle ne la recopie pas d'une table. Si la distance d'un jeu à l'autre était *la somme indépendante des distances de chaque côté* — un théorème que le §8 établira par énumération exhaustive — alors la distribution depuis n'importe quel jeu devrait être la **convolution de cette distribution d'un côté avec elle-même**. Vérifions la prédiction *avant* d'établir le théorème qui la justifie.

In [16]:
{
// Distribution des longueurs d'un côté : les nombres d'inversions sur S4.

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn voisins_nommes(g: Jeu) -> Vec<(Jeu, String)> {
    let (r, c) = g;
    let mut out: Vec<(Jeu, String)> = Vec::new();
    for k in [1usize, 2, 3] {
        out.push(((swap_valeurs_adjacentes(r, k), c), format!("R{}{}", k, k + 1)));
    }
    for k in [1usize, 2, 3] {
        out.push(((r, swap_valeurs_adjacentes(c, k)), format!("C{}{}", k, k + 1)));
    }
    out
}

fn bfs_jeu(depart: Jeu) -> BTreeMap<Jeu, usize> {
    let mut dist = BTreeMap::new();
    dist.insert(depart, 0);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for (v, _) in voisins_nommes(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let mut permset: BTreeSet<Tab> = BTreeSet::new();
let mut perm: Vec<usize> = vec![1, 2, 3, 4];
permset.insert((1, 2, 3, 4));
// Generateur : algorithme de Heap (= itertools.permutations, meme ensemble)
fn heap(k: usize, a: &mut Vec<usize>, out: &mut BTreeSet<Tab>) {
    if k == 1 {
        out.insert((a[0], a[1], a[2], a[3]));
        return;
    }
    for i in 0..k {
        heap(k - 1, a, out);
        if k % 2 == 0 {
            a.swap(i, k - 1);
        } else {
            a.swap(0, k - 1);
        }
    }
}
heap(4, &mut perm, &mut permset);
let perms: Vec<Tab> = permset.into_iter().collect(); // trie lexicographique = ordre itertools

let classic: Vec<(&str, Jeu)> = vec![
    ("Dilemme", ((3, 1, 4, 2), (3, 4, 1, 2))),
    ("Chicken", ((3, 2, 4, 1), (3, 4, 2, 1))),
    ("Deadlock", ((2, 1, 4, 3), (2, 4, 1, 3))),
    ("Chasse au cerf", ((4, 1, 3, 2), (4, 3, 1, 2))),
    ("Harmonie", ((4, 3, 2, 1), (4, 2, 3, 1))),
    ("Pennies", ((4, 1, 2, 3), (1, 4, 3, 2))),
];
let cherche = |nom: &str| -> Jeu { classic.iter().find(|(n, _)| *n == nom).unwrap().1 };

let dist_pd = bfs_jeu(cherche("Dilemme"));

let mut distribution: BTreeMap<usize, usize> = BTreeMap::new();
for &d in dist_pd.values() {
    *distribution.entry(d).or_default() += 1;
}

let inversions = |t: Tab| -> usize {
    let v = tab_vals(t);
    let mut n = 0;
    for i in 0..4 {
        for j in i + 1..4 {
            if v[i] > v[j] {
                n += 1;
            }
        }
    }
    n
};
let mut dist_inv: BTreeMap<usize, usize> = BTreeMap::new();
for &p in &perms {
    *dist_inv.entry(inversions(p)).or_default() += 1;
}
let un_cote: Vec<usize> = (0..7).map(|k| dist_inv[&k]).collect();
let somme40: usize = dist_inv.values().sum();
println!(
    "distribution des nombres d'inversions sur S4 : {:?} (somme : {} permutations )",
    un_cote, somme40
);
let convol = |u: &[usize], v: &[usize]| -> Vec<usize> {
    let mut w = vec![0usize; u.len() + v.len() - 1];
    for (i, &a) in u.iter().enumerate() {
        for (j, &b) in v.iter().enumerate() {
            w[i + j] += a * b;
        }
    }
    w
};
let predite = convol(&un_cote, &un_cote);
let mesuree: Vec<usize> = (0..13).map(|d| distribution[&d]).collect();
println!("convolution (un cote * un cote)  : {:?}", predite);
println!("distribution mesurée             : {:?}", mesuree);
println!("coïncidence exacte : {}", py_bool(predite == mesuree));
}


distribution des nombres d'inversions sur S4 : [1, 3, 5, 6, 5, 3, 1] (somme : 24 permutations )


convolution (un cote * un cote)  : [1, 6, 19, 42, 71, 96, 106, 96, 71, 42, 19, 6, 1]


distribution mesurée             : [1, 6, 19, 42, 71, 96, 106, 96, 71, 42, 19, 6, 1]


coïncidence exacte : True


()

## 8. Le théorème de décomposition

La coïncidence exacte de la cellule précédente n'est pas un hasard : c'est la trace d'un théorème. Énonçons-le d'abord informellement.

> **Décomposition.** La distance de swap entre deux jeux $G = (r_G, c_G)$ et $H = (r_H, c_H)$ est la **somme des distances de chaque côté** :
> $$d(G, H) = \ell(r_G, r_H) + \ell(c_G, c_H)$$
> où $\ell(a, b)$ est la **longueur de Coxeter** de la relabellisation qui envoie $a$ sur $b$ : le nombre d'inversions de la permutation relative $b \circ a^{-1}$, c'est-à-dire le nombre minimal de transpositions de valeurs adjacentes qui transforment $a$ en $b$.

L'intuition : les générateurs Ligne et Colonne sont *indépendants* — un chemin est un entrelacement d'un mot côté Ligne et d'un mot côté Colonne, et chaque côté se comporte comme le graphe de Cayley de $S_4$ muni des trois transpositions adjacentes, dont la distance est exactement la longueur de Coxeter. Nous n'affirmerons pas cette preuve conceptuelle comme établie (elle est rapportée en dette) — mais nous pouvons faire mieux que la croire : **vérifier l'énoncé sur chacune des 331 776 paires de jeux**, en comparant le BFS (la mesure) à la formule (la prédiction), indépendamment l'une de l'autre.

In [17]:
{
// === Section 8 : longueurs de Coxeter d'un cote ===

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn coxeter(a: Tab, b: Tab) -> usize {
    let av = tab_vals(a);
    let bv = tab_vals(b);
    let mut pos = [0usize; 5];
    for (i, &v) in av.iter().enumerate() {
        pos[v] = i;
    }
    let pi = [bv[pos[1]], bv[pos[2]], bv[pos[3]], bv[pos[4]]];
    let mut n = 0;
    for i in 0..4 {
        for j in i + 1..4 {
            if pi[i] > pi[j] {
                n += 1;
            }
        }
    }
    n
}

println!("Ligne : (3,1,4,2) -> (3,2,4,1), coxeter = {}", coxeter((3, 1, 4, 2), (3, 2, 4, 1)));
println!("Colonne : (3,4,1,2) -> (3,4,2,1), coxeter = {}", coxeter((3, 4, 1, 2), (3, 4, 2, 1)));
println!("retournement (1,2,3,4) -> (4,3,2,1), coxeter = {}", coxeter((1, 2, 3, 4), (4, 3, 2, 1)));
println!("identite (1,2,3,4) -> (1,2,3,4), coxeter = {}", coxeter((1, 2, 3, 4), (1, 2, 3, 4)));
}


Ligne : (3,1,4,2) -> (3,2,4,1), coxeter = 1


Colonne : (3,4,1,2) -> (3,4,2,1), coxeter = 1


retournement (1,2,3,4) -> (4,3,2,1), coxeter = 6


identite (1,2,3,4) -> (1,2,3,4), coxeter = 0


()

### 8.1 Vérification exhaustive sur les 331 776 paires

Les exemples ci-dessus portent sur trois paires choisies ; l'énoncé, lui, porte sur **toutes**. La vérification est directe et bornée : pour chacune des 576 sources, un parcours en largeur complet donne les distances mesurées, et la formule de Coxeter donne les distances prédites — 576 × 576 = 331 776 paires, dont aucune ne doit diverger.


In [18]:
{
// === Section 8.1 : verification exhaustive sur les 331 776 paires ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

use std::time::Instant;

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn voisins_nommes(g: Jeu) -> Vec<(Jeu, String)> {
    let (r, c) = g;
    let mut out: Vec<(Jeu, String)> = Vec::new();
    for k in [1usize, 2, 3] {
        out.push(((swap_valeurs_adjacentes(r, k), c), format!("R{}{}", k, k + 1)));
    }
    for k in [1usize, 2, 3] {
        out.push(((r, swap_valeurs_adjacentes(c, k)), format!("C{}{}", k, k + 1)));
    }
    out
}

fn bfs_jeu(depart: Jeu) -> BTreeMap<Jeu, usize> {
    let mut dist = BTreeMap::new();
    dist.insert(depart, 0);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for (v, _) in voisins_nommes(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn coxeter(a: Tab, b: Tab) -> usize {
    let av = tab_vals(a);
    let bv = tab_vals(b);
    let mut pos = [0usize; 5];
    for (i, &v) in av.iter().enumerate() {
        pos[v] = i;
    }
    let pi = [bv[pos[1]], bv[pos[2]], bv[pos[3]], bv[pos[4]]];
    let mut n = 0;
    for i in 0..4 {
        for j in i + 1..4 {
            if pi[i] > pi[j] {
                n += 1;
            }
        }
    }
    n
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let mut permset: BTreeSet<Tab> = BTreeSet::new();
let mut perm: Vec<usize> = vec![1, 2, 3, 4];
permset.insert((1, 2, 3, 4));
// Generateur : algorithme de Heap (= itertools.permutations, meme ensemble)
fn heap(k: usize, a: &mut Vec<usize>, out: &mut BTreeSet<Tab>) {
    if k == 1 {
        out.insert((a[0], a[1], a[2], a[3]));
        return;
    }
    for i in 0..k {
        heap(k - 1, a, out);
        if k % 2 == 0 {
            a.swap(i, k - 1);
        } else {
            a.swap(0, k - 1);
        }
    }
}
heap(4, &mut perm, &mut permset);
let perms: Vec<Tab> = permset.into_iter().collect(); // trie lexicographique = ordre itertools

let mut jeux31: Vec<Jeu> = Vec::new();
for &r in &perms {
    for &c in &perms {
        jeux31.push((r, c));
    }
}

let classic: Vec<(&str, Jeu)> = vec![
    ("Dilemme", ((3, 1, 4, 2), (3, 4, 1, 2))),
    ("Chicken", ((3, 2, 4, 1), (3, 4, 2, 1))),
    ("Deadlock", ((2, 1, 4, 3), (2, 4, 1, 3))),
    ("Chasse au cerf", ((4, 1, 3, 2), (4, 3, 1, 2))),
    ("Harmonie", ((4, 3, 2, 1), (4, 2, 3, 1))),
    ("Pennies", ((4, 1, 2, 3), (1, 4, 3, 2))),
];
let cherche = |nom: &str| -> Jeu { classic.iter().find(|(n, _)| *n == nom).unwrap().1 };

let dist_pd = bfs_jeu(cherche("Dilemme"));

let t44 = Instant::now();
let mut matrice: BTreeMap<Jeu, BTreeMap<Jeu, usize>> = BTreeMap::new();
let mut coherence = true;
'outer: for g in &jeux31 {
    let dg = bfs_jeu(*g);
    for h in &jeux31 {
        let formule = coxeter(g.0, h.0) + coxeter(g.1, h.1);
        if dg[h] != formule {
            coherence = false;
            println!("CONTRE-EXEMPLE : {:?} {:?} {} != {}", g, h, dg[h], formule);
            matrice.insert(*g, dg);
            break 'outer;
        }
    }
    matrice.insert(*g, dg);
}
let elapsed = t44.elapsed().as_secs_f64();
let paires: usize = matrice.values().map(|m| m.len()).sum();
println!("paires vérifiées : {} / 331776 en {:.1}s", paires, elapsed);
println!("décomposition d = coxeter(ligne) + coxeter(colonne) : {}", py_bool(coherence));
let mut excentricites: BTreeMap<usize, usize> = BTreeMap::new();
for dg in matrice.values() {
    *excentricites.entry(*dg.values().max().unwrap()).or_default() += 1;
}
println!("excentricités de tous les jeux : {:?}", excentricites);
let spot = matrice[&cherche("Dilemme")][&cherche("Chicken")];
println!(
    "spot-check matrice [Dilemme][Chicken] = {} (BFS §6 : {} )",
    spot,
    dist_pd[&cherche("Chicken")]
);
}


paires vérifiées : 331776 / 331776 en 1.9s


décomposition d = coxeter(ligne) + coxeter(colonne) : True


excentricités de tous les jeux : {12: 576}


spot-check matrice [Dilemme][Chicken] = 2 (BFS §6 : 2 )


()

### Interprétation : le théorème établi par énumération

**La décomposition vaut pour chacune des 331 776 paires.** Comme le théorème de la condition de transformation dans GT-3d (vérifié sur les 3 456 paires jeu-générateur), l'énoncé est ici *établi par énumération exhaustive* : l'espace est fini, la vérification est complète, et le résultat se lit sur la sortie. La preuve conceptuelle — indépendance des deux côtés, graphe de Cayley de $S_4$, produit cartésien — est **rapportée en dette** (§Sources) : nous n'en livrons pas la démonstration générale, seulement l'évidence exhaustive sur cet espace.

Trois corollaires immédiats, tous mesurés :

- **le diamètre vaut 12** = deux fois le maximum de la longueur de Coxeter d'un côté (6, le retournement) — cohérent avec la mesure du §2.2 ;
- **chaque jeu a une excentricité de 12** : l'espace est parfaitement isotrope, aucun point n'est plus central qu'un autre ; le Dilemme n'occupe aucune position privilégiée — c'est une propriété de *l'espace*, pas du jeu ;
- **la distribution depuis n'importe quel jeu** est la convolution de la distribution d'inversions d'un côté avec elle-même (cellule précédente) — ce qui explique la symétrie parfaite autour de 6.

Un point reste ouvert dans la géographie : si les *distances* sont simples, que dire du **nombre** de plus courts chemins ? C'est l'objet du §9.

## 9. Multiplicités : combien de plus courts chemins

La distance dit *combien de pas* ; elle ne dit pas *par où*. Deux jeux à même distance peuvent être reliés par un nombre très différent de chemins minimaux. Comptons-les en énumérant, pour une cible donnée, tous les chemins qui descendent strictement d'un niveau par étape (tout plus court chemin est une telle descente monotone).

In [19]:
{
// === Section 9 : multiplicites des plus courts chemins (voisinage interlace) ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn voisins_nommes(g: Jeu) -> Vec<(Jeu, String)> {
    let (r, c) = g;
    let mut out: Vec<(Jeu, String)> = Vec::new();
    for k in [1usize, 2, 3] {
        out.push(((swap_valeurs_adjacentes(r, k), c), format!("R{}{}", k, k + 1)));
    }
    for k in [1usize, 2, 3] {
        out.push(((r, swap_valeurs_adjacentes(c, k)), format!("C{}{}", k, k + 1)));
    }
    out
}

fn bfs_jeu(depart: Jeu) -> BTreeMap<Jeu, usize> {
    let mut dist = BTreeMap::new();
    dist.insert(depart, 0);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for (v, _) in voisins_nommes(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

fn plus_courts_chemins(depart: Jeu, arrivee: Jeu) -> Vec<Vec<String>> {
    // Cellule 47 : BFS separe (memes distances), MAIS les voisins de l'enumération
    // sont nommes EN INTERLACE : Rk, Ck, Rk, Ck, Rk, Ck (un seul for k, deux append).
    // C'est cet ordre qui fixe le PREMIER chemin affiche (ex : Pennies C12…, Harmonie R12 C12…).
    let d = bfs_jeu(depart);
    let n = d[&arrivee];
    let mut chemins: Vec<(Jeu, Vec<String>)> = vec![(depart, Vec::new())];
    for _ in 0..n {
        let mut next: Vec<(Jeu, Vec<String>)> = Vec::new();
        for (g, p) in &chemins {
            let dg = d[g];
            for k in [1usize, 2, 3] {
                let v: Jeu = (swap_valeurs_adjacentes(g.0, k), g.1);
                if d.get(&v).copied().unwrap_or(99) == dg + 1 {
                    let mut np = p.clone();
                    np.push(format!("R{}{}", k, k + 1));
                    next.push((v, np));
                }
                let v2: Jeu = (g.0, swap_valeurs_adjacentes(g.1, k));
                if d.get(&v2).copied().unwrap_or(99) == dg + 1 {
                    let mut np = p.clone();
                    np.push(format!("C{}{}", k, k + 1));
                    next.push((v2, np));
                }
            }
        }
        chemins = next;
    }
    chemins
        .into_iter()
        .filter(|(g, _)| *g == arrivee)
        .map(|(_, p)| p)
        .collect()
}

fn comb(n: usize, k: usize) -> usize {
    let mut r = 1usize;
    for i in 0..k {
        r = r * (n - i) / (i + 1);
    }
    r
}

let classic: Vec<(&str, Jeu)> = vec![
    ("Dilemme", ((3, 1, 4, 2), (3, 4, 1, 2))),
    ("Chicken", ((3, 2, 4, 1), (3, 4, 2, 1))),
    ("Deadlock", ((2, 1, 4, 3), (2, 4, 1, 3))),
    ("Chasse au cerf", ((4, 1, 3, 2), (4, 3, 1, 2))),
    ("Harmonie", ((4, 3, 2, 1), (4, 2, 3, 1))),
    ("Pennies", ((4, 1, 2, 3), (1, 4, 3, 2))),
];
let cherche = |nom: &str| -> Jeu { classic.iter().find(|(n, _)| *n == nom).unwrap().1 };

let dist_pd = bfs_jeu(cherche("Dilemme"));

let complement = |g: Jeu| -> Jeu {
    let f = |t: Tab| (5 - t.0, 5 - t.1, 5 - t.2, 5 - t.3);
    (f(g.0), f(g.1))
};
let anti = complement(cherche("Dilemme"));

for nom in ["Chicken", "Deadlock", "Chasse au cerf", "Pennies", "Harmonie"] {
    let cs = plus_courts_chemins(cherche("Dilemme"), cherche(nom));
    println!(
        "Dilemme -> {:16} distance {:2} : {:6} plus courts chemins (ex : {})",
        nom,
        dist_pd[&cherche(nom)],
        cs.len(),
        cs[0].join(" ")
    );
}
let n_anti = plus_courts_chemins(cherche("Dilemme"), anti).len();
let identite: Jeu = ((1, 2, 3, 4), (1, 2, 3, 4));
let retourne_ligne: Jeu = ((4, 3, 2, 1), (1, 2, 3, 4));
let mots_w0 = plus_courts_chemins(identite, retourne_ligne).len();
println!("\nDilemme -> anti-Dilemme : {} plus courts chemins", n_anti);
println!("mots réduits du retournement, un côté : {}", mots_w0);
println!(
    "entrelacements C(12,6) x 16 x 16 = {} x {} x {} = {}",
    comb(12, 6),
    mots_w0,
    mots_w0,
    comb(12, 6) * mots_w0 * mots_w0
);
}


Dilemme -> Chicken          distance  2 :      2 plus courts chemins (ex : R12 C12)


Dilemme -> Deadlock         distance  2 :      2 plus courts chemins (ex : R23 C23)


Dilemme -> Chasse au cerf   distance  2 :      2 plus courts chemins (ex : R34 C34)


Dilemme -> Pennies          distance  5 :     20 plus courts chemins (ex : C12 C23 C12 R34 R23)


Dilemme -> Harmonie         distance  6 :     80 plus courts chemins (ex : R12 C12 R34 R23 C34 C23)


Dilemme -> anti-Dilemme : 236544 plus courts chemins


mots réduits du retournement, un côté : 16


entrelacements C(12,6) x 16 x 16 = 924 x 16 x 16 = 236544


()

### Interprétation : de la commutation à l'explosion combinatoire

Le spectre des multiplicités est vaste : **exactement 2** chemins minimaux vers chacun des trois cousins (les deux étapes — une par côté — commutent, donc les deux ordres conviennent, et rien d'autre), **20** vers Pennies, **80** vers l'Harmonie, et **236 544** vers l'anti-Dilemme. Ce dernier compte se factorise exactement : $\binom{12}{6} \times 16 \times 16$ — les 16 *mots réduits* du retournement d'un côté (mesurés par la même énumération, restreinte à un seul côté), au carré pour les deux côtés, entrelacés dans $\binom{12}{6}$ ordres possibles. Le chemin minimal vers l'antagoniste absolu n'est donc pas une rareté : il est abondant, presque *tout chemin glouton qui descend monotone y conduit*.

La leçon structurelle : la distance (§6) est une fonction simple et additive, mais la **géométrie fine** des géodésiques porte l'information combinatoire — le nombre de mots réduits d'un élément de $S_4$ est une donnée non triviale de la théorie des groupes de Coxeter, rapportée en dette comme le reste du vocabulaire. C'est précisément quand les objets deviennent abondants et faciles à générer que la question du *certificat* devient pressante : qui garantit qu'un chemin proposé est correct ? C'est le §10.

## 10. Générateur contre certificat

Récapitulons les deux rôles, qui resteront séparés jusqu'au bout :

- le **générateur** (Rust, ce notebook) *explore* : BFS, énumération des chemins, comptages. Il **propose** des chemins minimaux — mais un programme qui s'exécute n'est pas une preuve qu'il calcule ce qu'il prétend calculer ;
- le **certificat** (Lean, `game_theory_lean/Swaps/Basic.lean`, écrit pour ce grain) **garantit** : qu'un chemin est bien formé (chaque étape est l'un des six générateurs — c'est *structurel*, le type des étapes ne contient rien d'autre), qu'il mène bien du départ à l'arrivée (évalué par le noyau Lean, `rfl`), et — sur le cas témoin — qu'aucun chemin plus court n'existe (énumération décidable).

Commençons par le miroir Rust du vérificateur, pour voir *ce qui se joue* : un chemin **valide** (il arrive) n'est pas un chemin **minimal** (il arrive le plus vite possible).

In [20]:
{
// === Section 10 : appliquer un chemin nomme, valider un certificat ===

use std::collections::{BTreeMap, BTreeSet, VecDeque};

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);

fn tab_vals(t: Tab) -> Vec<usize> {
    vec![t.0, t.1, t.2, t.3]
}

fn swap_valeurs_adjacentes(t: Tab, k: usize) -> Tab {
    let mut v = tab_vals(t);
    let pk = v.iter().position(|&x| x == k).unwrap();
    let pk1 = v.iter().position(|&x| x == k + 1).unwrap();
    v.swap(pk, pk1);
    (v[0], v[1], v[2], v[3])
}

fn voisins_nommes(g: Jeu) -> Vec<(Jeu, String)> {
    let (r, c) = g;
    let mut out: Vec<(Jeu, String)> = Vec::new();
    for k in [1usize, 2, 3] {
        out.push(((swap_valeurs_adjacentes(r, k), c), format!("R{}{}", k, k + 1)));
    }
    for k in [1usize, 2, 3] {
        out.push(((r, swap_valeurs_adjacentes(c, k)), format!("C{}{}", k, k + 1)));
    }
    out
}

fn bfs_jeu(depart: Jeu) -> BTreeMap<Jeu, usize> {
    let mut dist = BTreeMap::new();
    dist.insert(depart, 0);
    let mut q = VecDeque::new();
    q.push_back(depart);
    while let Some(u) = q.pop_front() {
        let du = dist[&u];
        for (v, _) in voisins_nommes(u) {
            if !dist.contains_key(&v) {
                dist.insert(v, du + 1);
                q.push_back(v);
            }
        }
    }
    dist
}

// str(list) de Python pour les listes de chaines : ['R12', 'C12']
fn py_list_str(v: &[&str]) -> String {
    let parts: Vec<String> = v.iter().map(|s| format!("'{}'", s)).collect();
    format!("[{}]", parts.join(", "))
}

fn py_bool(b: bool) -> &'static str {
    if b { "True" } else { "False" }
}

let classic: Vec<(&str, Jeu)> = vec![
    ("Dilemme", ((3, 1, 4, 2), (3, 4, 1, 2))),
    ("Chicken", ((3, 2, 4, 1), (3, 4, 2, 1))),
    ("Deadlock", ((2, 1, 4, 3), (2, 4, 1, 3))),
    ("Chasse au cerf", ((4, 1, 3, 2), (4, 3, 1, 2))),
    ("Harmonie", ((4, 3, 2, 1), (4, 2, 3, 1))),
    ("Pennies", ((4, 1, 2, 3), (1, 4, 3, 2))),
];
let cherche = |nom: &str| -> Jeu { classic.iter().find(|(n, _)| *n == nom).unwrap().1 };

let dist_pd = bfs_jeu(cherche("Dilemme"));

let applique_etape = |g: Jeu, e: &str| -> Jeu {
    let k: usize = e[1..2].parse().unwrap();
    if e.starts_with('R') {
        (swap_valeurs_adjacentes(g.0, k), g.1)
    } else {
        (g.0, swap_valeurs_adjacentes(g.1, k))
    }
};
let applique_chemin = |g0: Jeu, chemin: &[&str]| -> Jeu {
    let mut g = g0;
    for e in chemin {
        g = applique_etape(g, e);
    }
    g
};
let (pd50, chicken50) = (cherche("Dilemme"), cherche("Chicken"));
let c_minimal: Vec<&str> = vec!["R12", "C12"];
let c_detour: Vec<&str> = vec!["R12", "R23", "R23", "C12"];
println!(
    "chemin minimal   {} : valide = {}",
    py_list_str(&c_minimal),
    py_bool(applique_chemin(pd50, &c_minimal) == chicken50)
);
println!(
    "chemin détour    {} : valide = {}",
    py_list_str(&c_detour),
    py_bool(applique_chemin(pd50, &c_detour) == chicken50)
);
println!();
println!(
    "longueurs : {} vs {} | distance BFS : {}",
    c_minimal.len(),
    c_detour.len(),
    dist_pd[&chicken50]
);
}


chemin minimal   ['R12', 'C12'] : valide = True


chemin détour    ['R12', 'R23', 'R23', 'C12'] : valide = True


longueurs : 2 vs 4 | distance BFS : 2


()

### Interprétation : valide n'est pas minimal

Le certificat d'arrivée accepte les **deux** chemins — et il a raison : les deux mènent bien du Dilemme à Chicken (le détour insère un aller-retour $R_{23} \circ R_{23}$, qui s'annule). Mais seul le premier est minimal : la longueur 4 du détour dépasse la distance 2. Un vérificateur d'arrivée est donc *complet mais laxiste* ; discriminer les chemins minimaux exige soit l'exploration exhaustive (ce que fait le BFS), soit une borne inférieure (ce que fait le module Lean sur le cas témoin : aucun chemin de longueur < 2 n'existe, par énumération décidable des cas).

C'est la division du travail que le grain installe : **le générateur produit des candidats abondants** (§9 : des centaines de milliers vers l'anti-Dilemme), **le certificat tranche** — arrivée garantie toujours, minimalité garantie sur le témoin. Voyons le module Lean lui-même, cité aux lignes mesurées du fichier.

## 10-bis. Le module Swaps/Basic.lean, cité aux lignes mesurées

Le module compagnon vit dans `game_theory_lean/Swaps/Basic.lean` (même lake que les preuves Arrow et Shapley). Il est **volontairement sans Mathlib** : tout y est calcul fini décidable sur des listes littérales — les théorèmes se closent par `rfl` (évaluation par le noyau) ou `decide` (énumération décidable), sans tactique sophistiquée ni axiome. La cellule suivante lit le fichier depuis ce clone et imprime chaque définition et chaque théorème à sa ligne réelle — les citations du reste de la section s'appuient sur ces numéros, pas sur une mémoire.

In [21]:
{
// === Section 10-bis : lecture du module Swaps/Basic.lean (std::fs) ===

// Cellule 53 : localiser Swaps/Basic.lean depuis le repertoire courant ou ses parents
fn trouver_swaps_lean() -> Option<std::path::PathBuf> {
    let mut bases: Vec<std::path::PathBuf> = Vec::new();
    if let Ok(cwd) = std::env::current_dir() {
        bases.push(cwd.clone());
        let mut cur = cwd;
        while let Some(p) = cur.parent() {
            bases.push(p.to_path_buf());
            cur = p.to_path_buf();
        }
    }
    for base in &bases {
        for rel in [
            "game_theory_lean/Swaps/Basic.lean",
            "MyIA.AI.Notebooks/GameTheory/game_theory_lean/Swaps/Basic.lean",
        ] {
            let cible = base.join(rel);
            if cible.exists() {
                return Some(cible);
            }
        }
    }
    None
}

fn first100(s: &str) -> String {
    s.chars().take(100).collect()
}

let chemin_module = trouver_swaps_lean();
match chemin_module {
    None => println!("module Swaps/Basic.lean introuvable depuis ce répertoire — cloner le dépôt complet pour la lecture des citations"),
    Some(p) => {
        let src = std::fs::read_to_string(&p).unwrap();
        let lignes: Vec<&str> = src.lines().collect();
        println!("{} : {} lignes", p.file_name().unwrap().to_string_lossy(), lignes.len());
        println!();
        for (i, l) in lignes.iter().enumerate() {
            if l.starts_with("def ")
                || l.starts_with("theorem ")
                || l.starts_with("inductive ")
                || l.starts_with("structure ")
            {
                println!("  {:4}: {}", i + 1, first100(l.trim_end()));
            }
        }
        println!();
        println!("occurrences de 'sorry' dans le module : {}", src.matches("sorry").count());
    }
}
let _ = ();
}


Basic.lean : 135 lignes


    34: def Table : Type := List Nat


    37: def Jeu : Type := Table × Table


    46: def swapAdj (t : Table) (k : Nat) : Table :=


    50: inductive Etape where


    56: def Etape.k : Etape → Nat


... (11 outputs left out)

   130: def cheminValide (depart arrivee : Jeu) (p : List Etape) : Prop :=


   135: theorem certificat_cerf : cheminValide dilemme cerf [.R34, .C34] := by rfl


occurrences de 'sorry' dans le module : 0


()

### Interprétation : ce que chaque théorème garantit

La lecture du module aligne les garanties, dans l'ordre où le notebook les a rencontrées :

- **`certificat_chemin`** — `applique dilemme cheminDilemmeChicken = chicken`, clos par `rfl` : le chemin `[R12, C12]` proposé par le BFS mène bien du Dilemme à Chicken. C'est le certificat d'arrivée, garanti par évaluation du noyau Lean — pas par la ré-exécution d'un programme Python, mais par la réduction du terme lui-même.
- **`chemin_valide_non_minimal`** — le détour `[R12, R23, R23, C12]` arrive aussi (la cellule miroir l'a mesuré côté Rust) : le module Lean l'admet explicitement, pour montrer que le certificat d'arrivée n'exclut pas les détours.
- **`aucun_chemin_court`** — aucun chemin de longueur 0 ou 1 ne relie le Dilemme à Chicken : longueur 0 rejetée par `decide` (les jeux diffèrent), longueur 1 par `cases e <;> decide` — les six générateurs sont énumérés un à un, chacun tranché par décision. C'est la **borne inférieure**.
- **`distance_dilemme_chicken`** — la conjonction : un chemin de longueur 2 existe *et* aucun plus court n'existe. La distance vaut **exactement 2** — énoncé complet, prouvé.

La minimalité générale (pour deux jeux quelconques) reste du côté du générateur : elle exige l'exploration. Le module Lean la certifie sur le témoin parce que l'espace des chemins de longueur < 2 est assez petit pour l'énumération décidable — c'est exactement la frontière où le certificat devient abordable, et le grain la marque explicitement plutôt que de la masquer. Le build du module (`lake build Swaps`) est sans dépendance Mathlib : il se compile en secondes, aucune preuve du lake n'est touchée.

## Exercices

### Exercice 1 — Énumérer les murs par codimension

Reproduisez et étendez le comptage du §1 : énumérez les 75 ordres faibles, comptez-les par codimension, puis **par type de blocs** (tailles décroissantes). Vérifiez que la somme des comptes par type redonne 75, et que chaque type de mur $(2,1,1)$, $(2,2)$, $(3,1)$, $(4)$ est non vide.

**Indice** : la signature de blocs s'obtient avec `Counter` ; la codimension est `4 - len(set(t))`. Pour le compte par type, regroupez par `tuple(sorted(Counter(t).values(), reverse=True))`.

**Étape 1** : énumérer les 75 clés canoniques (§1.1). **Étape 2** : grouper par signature. **Étape 3** : afficher le compte par signature et vérifier la somme.

In [22]:
{
// Exercice 1 : enumeration des murs par codimension et par type de blocs

// Exercice 1 : enumeration des murs par codimension et par type de blocs
// TODO etudiant : reproduire la table du 1.2 (75 ordres faibles, 4 signatures de murs)
// Indice : signature = tuple(sorted(Counter(t).values(), reverse=True)) -- en Rust, effectifs par valeur, tries decroissant
use std::collections::BTreeMap;
let resultat_ex1: BTreeMap<Vec<usize>, usize> = BTreeMap::new();  // TODO etudiant : {signature: nombre d'ordres faibles}
println!("Exercice 1 : a completer (compte par type de blocs)");
}


Exercice 1 : a completer (compte par type de blocs)


()

### Exercice 2 — Traverser : exhiber un triplet complet

Choisissez un jeu strict qui n'est **pas** le PD (par exemple le jeu de la poule mouillée — vérifiez son encodage ordinal avant de l'utiliser), une de ses facettes, et exhibez le triplet **(chambre A, mur, chambre B)** complet avec les trois matrices, comme au §3.1.

**Indice** : une facette = une paire de valeurs adjacentes $(k, k+1)$ ; le mur s'obtient par `faire_le_tie(table, k)` ; la chambre B par `swap_valeurs_adjacentes(table, k)` ; vérifiez enfin que `briser_le_tie(mur)` contient bien les deux chambres.

**Étape 1** : poser les tables du jeu choisi. **Étape 2** : choisir $k$ et construire le mur et la chambre B. **Étape 3** : vérifier la double incidence avec `briser_le_tie`.

In [23]:
{
// Exercice 2 : exhiber chambre -> mur -> chambre pour un autre jeu

// Exercice 2 : exhiber chambre -> mur -> chambre pour un autre jeu
// TODO etudiant : remplir les trois matrices et la verification
type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);
let chambre_A: Option<Jeu> = None;  // TODO etudiant : (row, col) du jeu strict choisi
let mur_ex2: Option<Jeu> = None;    // TODO etudiant : le mur traverse (via faire_le_tie)
let chambre_B: Option<Jeu> = None;  // TODO etudiant : la chambre d'arrivee (via swap_valeurs_adjacentes)
println!("Exercice 2 : a completer (triplet chambre-mur-chambre)");
}


Exercice 2 : a completer (triplet chambre-mur-chambre)


()

### Exercice 3 — Make tie / break tie : vers les archétypes

Depuis le jeu **Stag Hunt** de GT-21 (Ligne $(4,1,3,2)$, Colonne $(4,3,1,2)$), construisez par `make_tie` un mur de chaque type : un mur Ligne, un mur Colonne, puis un double mur (codim 2). Pour chacun, appliquez `break_tie` dans les deux sens et observez les deux chambres de sortie — vous venez de cartographier le voisinage d'un jeu de coordination.

**Indice** : le double mur s'obtient en composant deux `make_tie` successifs (comme au §4.2 pour la coordination). Les deux sens de `break_tie` doivent redonner deux jeux **stricts** distincts.

**Étape 1** : construire les trois murs. **Étape 2** : pour chacun, briser dans les deux sens. **Étape 3** : vérifier que chaque paire de sorties est distincte et stricte.

In [24]:
{
// Exercice 3 : voisinage de Stag Hunt par make_tie / break_tie

// Exercice 3 : voisinage de Stag Hunt par make_tie / break_tie
// TODO etudiant : les trois murs (ligne, colonne, double) et leurs sorties
use std::collections::BTreeMap;
type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);
let murs_ex3: Option<BTreeMap<&str, Jeu>> = None;     // TODO etudiant : {nom_du_mur: jeu_mur}
let sorties_ex3: Option<BTreeMap<&str, (Jeu, Jeu)>> = None;  // TODO etudiant : {nom_du_mur: (sortie_haut, sortie_bas)}
println!("Exercice 3 : a completer (voisinage de Stag Hunt)");
}


Exercice 3 : a completer (voisinage de Stag Hunt)


()

### Exercice 4 — Distance par sphères, sans reconstruire tout le BFS

La fonction `bfs` du §6 calcule les distances vers *tous* les jeux. Pour une seule cible, c'est du gaspillage — et parfois impossible si l'espace était plus grand. Écrivez `distance_par_spheres(depart, arrivee)` qui n'explore que les sphères successives jusqu'à rencontrer l'arrivée : maintenez l'ensemble des jeux du niveau courant, engendrez le niveau suivant par générateurs, arrêtez dès que l'arrivée apparaît, et renvoyez la distance atteinte. La fonction doit renvoyer le même verdict que le BFS complet pour toute cible — testez-la au moins sur Chicken (distance 2) et sur l'anti-Dilemme (distance 12).

In [25]:
{
// Exercice 4 : distance par spheres, sans reconstruire tout le BFS

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);
let dilemme: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let chicken: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));

fn distance_par_spheres(_depart: Jeu, _arrivee: Jeu) -> Option<usize> {
    // Distance de swap par exploration des spheres successives seulement.
    // Etape 1 : niveau courant = [depart], distance = 0
    // Etape 2 : engendrer le niveau suivant par les six generateurs
    // Etape 3 : si l'arrivee y figure, renvoyer la distance ; sinon recommencer
    // TODO etudiant
    None  // a remplacer
}

let resultat_ex4 = distance_par_spheres(dilemme, chicken);
// Indice : la reponse attendue est petite ; le chemin de cette longueur est
// exhibe au §6 et certifie par Swaps/Basic.lean (distance exacte 2).
}


()

### Exercice 5 — Le diamètre depuis une autre source : le chiffre, pas l'impression

Le §8 a mesuré que *tous* les jeux ont une excentricité de 12. Vérifiez-le depuis une source différente du Dilemme : écrivez `excentricite_depuis(source)` (distance maximale de la source à tout autre jeu) et appliquez-la à **Pennies**. Puis identifiez **combien de jeux** sont à distance 12 de Pennies et lequel c'est — la réponse est un couple (chiffre, jeu), pas une impression.

In [26]:
{
// Exercice 5 : le diametre depuis une autre source

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);
let pennies: Jeu = ((4, 1, 2, 3), (1, 4, 3, 2));

fn excentricite_depuis(_source: Jeu) -> Option<usize> {
    // Excentricite d'un jeu : sa distance maximale a tout autre jeu.
    // Etape 1 : BFS complet depuis source
    // Etape 2 : renvoyer le maximum des distances
    // TODO etudiant
    None  // a remplacer
}

let resultat_ex5 = excentricite_depuis(pennies);
// Indice : le theoreme de decomposition (§8) predit la valeur sans
// aucun calcul de BFS -- la formule de Coxeter suffit. Les deux doivent coincider.
}


()

### Exercice 6 — Le certificat d'un chemin fourni : valide, et minimal ?

On vous fournit un chemin du Dilemme à Chicken : `["R12", "R23", "R23", "C12"]`. Écrivez `certificat(depart, arrivee, chemin)` qui renvoie un **couple de booléens** : (validité, minimalité).

- la **validité** se certifie en appliquant le chemin (§10, miroir Python) et en comparant l'arrivée — c'est ce que `Swaps/Basic.lean` garantit par `rfl` ;
- la **minimalité** exige de comparer la longueur du chemin à la distance réelle — soit par re-BFS (exercice 4), soit par la formule de Coxeter (§8).

Votre verdict sur le chemin fourni doit être `(True, False)` : il arrive, mais en 4 pas au lieu de 2. Puis produisez le certificat complet d'un chemin qui mérite les deux `True`.

In [27]:
{
// Exercice 6 : le certificat d'un chemin fourni

type Tab = (usize, usize, usize, usize);
type Jeu = (Tab, Tab);
let dilemme: Jeu = ((3, 1, 4, 2), (3, 4, 1, 2));
let chicken: Jeu = ((3, 2, 4, 1), (3, 4, 2, 1));
let chemin_fourni = ["R12", "R23", "R23", "C12"];   // un detour, fourni

fn certificat(_depart: Jeu, _arrivee: Jeu, _chemin: &[&str]) -> Option<(bool, bool)> {
    // Verdict d'un chemin : couple (validite, minimalite).
    // Etape 1 : validite -- appliquer le chemin, comparer l'arrivee
    // Etape 2 : minimalite -- la longueur egale-t-elle la distance ?
    // Renvoyer (bool, bool)
    // TODO etudiant
    None  // a remplacer
}

let resultat_ex6 = certificat(dilemme, chicken, &chemin_fourni);
// Indice : attendu (True, False) pour le chemin fourni ; un chemin minimal
// (§6) merite (True, True).
}


()

## Conclusion

**Ce que la première moitié établit (tout calculé, tout vérifié dans les sorties) :**

1. l'univers des jeux 2×2 **à rangs** compte $75^2 = 5625$ états, dont les 576 jeux stricts ne sont que les chambres ouvertes (10,2 %) — le reste est l'architecture : 1728 murs simples, 1968 strates de codim 2, jusqu'au jeu totalement plat (codim 6, unique) ;
2. l'incidence est exacte : chaque mur simple sépare **exactement 2 chambres**, chaque chambre a **exactement 3 facettes** par côté — et les 1728 arêtes du graphe des chambres coïncident mur pour mur avec les 1728 strates de codim 1 ;
3. le graphe des chambres est **connexe**, de diamètre 6 par côté (12 pour le jeu) — avec la distribution symétrique des distances $\{1,3,5,6,5,3,1\}$ ;
4. la grammaire des swaps de GT-21 se recompose : les 3 swaps adjacents sont des traversées de facettes (1 mur), les 3 autres des raccourcis — 3, 3 et 5 murs **mesurés** dans le graphe, et non lus sur la formule — avec $R(1,3) = R(1,2) \circ R(2,3) \circ R(1,2)$ vérifié ;
5. *make tie* / *break tie* sont implémentées, et l'aller-retour est vérifié sur **les 36 murs simples** d'un côté — deux chambres strictes distinctes, exactement les faces que `briser_le_tie` leur connaît : un jeu à égalités est un objet **biface**, pas une définition relâchée.

**Ce que la seconde moitié établit :**

- **La distance de swap est mesurable exactement** : BFS depuis n'importe quel jeu, sphères emboîtées. Depuis le Dilemme : Chicken, Deadlock et la chasse au cerf à distance 2 (chacun en échangeant une paire différente de rangs adjacents chez chaque joueur), Pennies à 5, Harmonie à 6, l'anti-Dilemme à 12.
- **Théorème de décomposition (établi par énumération exhaustive, 331 776 paires)** : $d(G, H) = \ell(r_G, r_H) + \ell(c_G, c_H)$, la somme des longueurs de Coxeter des permutations relatives. Corollaires mesurés : diamètre 12, excentricité 12 pour *tous* les jeux (isotropie parfaite), distribution depuis tout jeu = convolution de la distribution d'inversions d'un côté avec elle-même.
- **Les multiplicités portent la combinatoire** : de 2 chemins minimaux (commutation des deux côtés) vers chacun des trois cousins à 236 544 = $\binom{12}{6} \times 16^2$ vers l'anti-Dilemme — les géodésiques abondent exactement là où la distance est maximale.
- **Générateur contre certificat** : le BFS Rust propose, le module Lean `Swaps/Basic.lean` garantit — arrivée par `rfl`, minimalité par énumération décidable sur le témoin Dilemme → Chicken (distance exactement 2). Le module est sans Mathlib et sans `sorry` ; la minimalité générale reste du côté du générateur, et la frontière est explicitée.
- **Position dans la série** : les chambres et les murs sont **dans ce carnet** (§1-§4) ; les chemins — la métrique de l'espace entier — en sont la seconde moitié (§6-§10). Le maillon manquant (rapporté en dette) : le quotient par renommage et la structure $S_4 \times S_4$ comme *tore*, dont notre espace complet est le dépliage.

**Ce qui reste ouvert** — les cinq dettes de dérivation, à honorer depuis les sources primaires avant toute affirmation publique : le quotient $576 \to 144$ (facteur exact à re-dériver), l'action du groupe $S_4 \times S_4$, le tore à 37 trous, la preuve conceptuelle de la décomposition, la théorie générale des groupes de Coxeter derrière les mots réduits — et la vérification firsthand des trois arXiv. Ce carnet ne les affirme pas : il les porte comme dette (§Sources).

**La phrase à retenir** : *un objet botanique n'est pas un objet plus flou — c'est un objet de dimension inférieure depuis lequel plusieurs prolongements sont possibles ; et la distance entre deux chambres se mesure en murs traversés.*

*Chantier des jeux 2×2 — voir aussi* : [GT-3 Topology2x2](GameTheory-03-Topology2x2-Python.ipynb) (l'espace et la grammaire) · [Dynamics-10 Le Joueur LLM](Dynamics/Dynamics-10-Joueur-LLM-Python.ipynb) (la table périodique appliquée à un LLM) · [GT-3d Deux-Espèces-de-Flèches](GameTheory-03d-Deux-Especes-de-Fleches-Python.ipynb) (quand un swap préserve les équilibres) · [GT-3e Méta-Actions-Tarifées](GameTheory-03e-Meta-Actions-Tarifees-Python.ipynb) (le capstone).

## Sources et dettes

- **Bruns & Kimmich** — la description géométrique (chambres/murs/strates, make/break tie, archétypes) est **rapportée** de leur formalisation des jeux 2×2, non dérivée de la source primaire ici (dette n°4). Tout ce que le carnet affirme chiffré (75, 5625, 1728, 3, 2, 6, 12) est **calculé localement** et reproductible en ré-exécutant.
- **Robinson & Goforth** — le tableau périodique des jeux 2×2 ; ancêtre Rapoport & Guyer. (L'attribution « Gale et Morris » qui circule dans certains transcripts est une hallucination corrigée.)
- **Vocabulaire** : volontairement minimal — chambres, murs, strates, codimension, facettes, distance de swap, générateur, certificat. Le seul terme de groupe employé est celui **que le carnet calcule** : la longueur de Coxeter d'un swap (§3.2) et la somme de ces longueurs qui donne la distance (§8) ; la théorie générale des groupes de Coxeter reste **rapportée** en dette, jamais affirmée.

**Mesuré dans ce carnet** (chaque chiffre est l'output d'une cellule) : les 75 ordres faibles et les 5 625 jeux à rangs par codimension, l'incidence exacte chambre-mur, le graphe des chambres (connexité, diamètre, distribution), les longueurs de Coxeter des six swaps, l'aller-retour *make tie* / *break tie* ; puis les distances aux archétypes, la distribution complète, la vérification exhaustive de la décomposition (331 776 paires), les excentricités, les multiplicités des plus courts chemins, la factorisation $\binom{12}{6} \times 16^2$, le miroir Python du vérificateur et les citations de `Swaps/Basic.lean` aux lignes réelles.

**Dettes ouvertes (HARD), jamais affirmées** :

1. **Le quotient $576 \to 144$** (classes de jeux à renommage des stratégies près) et la structure $S_4 \times S_4$ comme tore — Robinson et Goforth rapportent cette lecture ; tout ce carnet travaille sur l'espace complet des 576 jeux et n'affirme rien du quotient.
2. **La preuve conceptuelle de la décomposition** (indépendance des côtés, graphe de Cayley, produit cartésien de graphes) — nous en livrons l'énumération exhaustive, pas la démonstration générale.
3. **Le tore à 37 trous** et la topologie du quotient — hors scope, rapporté.
4. **Les références arXiv** (2309.15981, 2102.00053, 1704.02230) — identités vérifiées en première main (API arXiv) : Tohmé & Viglizzo, *A categorical representation of games* · Czechowski & Piliouras, *Poincaré-Bendixson Limit Sets in Multi-Agent Learning* · Hedges, *Coherence for lenses and open games* ; la lecture intégrale des trois reste à faire.
5. **La théorie des mots réduits** (16 mots pour le retournement de $S_4$) — mesurée par énumération ; la théorie générale des groupes de Coxeter qui l'explique est rapportée.

**Module compagnon** : `game_theory_lean/Swaps/Basic.lean` — pur core Lean, sans Mathlib, sans `sorry` ; `lake build Swaps` sans dépendance.

**Encodages** : identiques à GameTheory-3 et GT-3d (tables de rangs 1-4, cellules dans l'ordre haut-gauche, haut-droit, bas-gauche, bas-droit).
